In [ ]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_cn7
from common.train_test_split import split_data
from common.model_composition import model_composition

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"../data/raw/moldset_labeled_cn7_restored.csv")

In [ ]:
X_cn7, y_cn7 = preprocessing_cn7(train_cn7)

print("X shape:", X_cn7.shape)
print("y shape:", y_cn7.shape)

# 1. 데이터 전처리

### 원본 복사 및 시간형 변환

In [ ]:
train_cn7_pre = train_cn7.copy()

train_cn7_pre["TimeStamp"] = pd.to_datetime(
    train_cn7_pre["TimeStamp"],
    errors="coerce"
)

train_cn7_pre = train_cn7_pre.sort_values(
    ["TimeStamp", "RH_LH"]
).reset_index(drop=True)

train_cn7_pre.head()

### 결측치 확인

In [ ]:
train_cn7_pre.isnull().sum()

### 상수 컬럼 제거

In [ ]:
train_cn7_pre = train_cn7_pre.drop(
    columns=["Clamp_Open_Position"]
)

### 원래 행 번호 제거

In [ ]:
train_cn7_pre = train_cn7_pre.drop(
    columns=["original_index"]
)

### part name 제거

In [ ]:
train_cn7_pre = train_cn7_pre.drop(
    columns=["PART_NAME"]
)

### RH LH 숫자로 변환

In [ ]:
train_cn7_pre["Side_RH"] = (
    train_cn7_pre["RH_LH"] == "RH"
).astype(int)

In [ ]:
train_cn7_pre[
    ["RH_LH", "Side_RH"]
].value_counts()

In [ ]:
train_cn7_pre = train_cn7_pre.drop(
    columns=["RH_LH"]
)

### timestamp는 모델 입력 변수와 그룹 변수로 분리

In [ ]:
groups_cn7 = train_cn7_pre["TimeStamp"].copy()

In [ ]:
train_cn7_pre = train_cn7_pre.drop(
    columns=["TimeStamp"]
)

In [ ]:
cn7_work = train_cn7_pre.copy()

cn7_work["TimeStamp"] = groups_cn7.values

cn7_work.head()

In [ ]:
cn7_work.columns.tolist()

### 센서컬럼 자동 지정

In [ ]:
SENSOR_COLS = [
    col for col in cn7_work.columns
    if col not in ["PassOrFail", "Side_RH", "TimeStamp"]
]

print("센서 변수 개수:", len(SENSOR_COLS))
print(SENSOR_COLS)

### cycle 단위 데이터 생성

In [ ]:
cn7_cycle = (
    cn7_work
    .sort_values("TimeStamp")
    .drop_duplicates(
        subset=["TimeStamp"],
        keep="first"
    )
    [["TimeStamp"] + SENSOR_COLS]
    .reset_index(drop=True)
)

print("전체 행 수:", len(cn7_work))
print("고유 Cycle 수:", len(cn7_cycle))

cn7_cycle.head()

### diff 파생번수 생성(생산중단 구간 부분)

In [ ]:
cn7_cycle["time_gap_sec"] = (
    cn7_cycle["TimeStamp"]
    .diff()
    .dt.total_seconds()
)

cn7_cycle["session_id"] = (
    cn7_cycle["time_gap_sec"]
    .gt(300)
    .cumsum()
)

cn7_cycle[
    ["TimeStamp", "time_gap_sec", "session_id"]
].head(10)

In [ ]:
for col in SENSOR_COLS:
    cn7_cycle[f"{col}_diff1"] = (
        cn7_cycle
        .groupby("session_id")[col]
        .diff()
    )

In [ ]:
diff_cols = [
    col for col in cn7_cycle.columns
    if col.endswith("_diff1")
]

print("diff 변수 개수:", len(diff_cols))

cn7_cycle[
    ["TimeStamp"] + diff_cols[:5]
].head()

#### 첫 cycle의 nan 처리

In [ ]:
cn7_cycle[diff_cols] = (
    cn7_cycle[diff_cols]
    .fillna(0)
)

In [ ]:
cn7_cycle[diff_cols].isnull().sum().sum()

### diff 1 병합

In [ ]:
cn7_work = cn7_work.merge(
    cn7_cycle[
        ["TimeStamp"] + diff_cols
    ],
    on="TimeStamp",
    how="left"
)

In [ ]:
print("cn7_work shape:", cn7_work.shape)

cn7_work[
    [
        "TimeStamp",
        "Side_RH",
        SENSOR_COLS[0],
        f"{SENSOR_COLS[0]}_diff1"
    ]
].head(10)

In [ ]:
print("전체 행 수:", len(cn7_work))

In [ ]:
cn7_work[diff_cols].isnull().sum().sum()

### rolling 생성

> 이동구간 통계

shift(1) -> 현재 제외, 직전 5개만 사용

In [ ]:
ROLL_WINDOW = 5

for col in SENSOR_COLS:
    # 직전 5개 Cycle 평균
    cn7_cycle[f"{col}_roll5_mean"] = (
        cn7_cycle
        .groupby("session_id")[col]
        .transform(
            lambda x: x.shift(1).rolling(
                window=ROLL_WINDOW,
                min_periods=1
            ).mean()
        )
    )

    # 직전 5개 Cycle 표준편차
    cn7_cycle[f"{col}_roll5_std"] = (
        cn7_cycle
        .groupby("session_id")[col]
        .transform(
            lambda x: x.shift(1).rolling(
                window=ROLL_WINDOW,
                min_periods=2
            ).std()
        )
    )

In [ ]:
roll_mean_cols = [
    col for col in cn7_cycle.columns
    if col.endswith("_roll5_mean")
]

roll_std_cols = [
    col for col in cn7_cycle.columns
    if col.endswith("_roll5_std")
]

print("Rolling Mean:", len(roll_mean_cols))
print("Rolling Std :", len(roll_std_cols))

In [ ]:
cn7_cycle[
    [
        "TimeStamp",
        "Injection_Time",
        "Injection_Time_diff1",
        "Injection_Time_roll5_mean",
        "Injection_Time_roll5_std"
    ]
].head(10)

In [ ]:
rolling_cols = roll_mean_cols + roll_std_cols

cn7_cycle[rolling_cols] = (
    cn7_cycle[rolling_cols]
    .fillna(0)
)

In [ ]:
cn7_cycle[rolling_cols].isnull().sum().sum()

In [ ]:
cn7_work = cn7_work.merge(
    cn7_cycle[
        ["TimeStamp"] + rolling_cols
    ],
    on="TimeStamp",
    how="left"
)

In [ ]:
print(cn7_work.shape)
print("행 수:", len(cn7_work))

In [ ]:
cn7_work[rolling_cols].isnull().sum().sum()

### deviation 생성

> 현재값 - 직전 5개 평균

현재 값이 최근 5개 평균에서 얼마나 벗어났는가

In [ ]:
for col in SENSOR_COLS:
    cn7_cycle[f"{col}_roll5_dev"] = (
        cn7_cycle[col]
        - cn7_cycle[f"{col}_roll5_mean"]
    )

In [ ]:
dev_cols = [
    col for col in cn7_cycle.columns
    if col.endswith("_roll5_dev")
]

print("Deviation 변수 개수:", len(dev_cols))
print(dev_cols[:5])

In [ ]:
cn7_cycle[
    [
        "TimeStamp",
        "Injection_Time",
        "Injection_Time_roll5_mean",
        "Injection_Time_roll5_dev"
    ]
].head(10)

In [ ]:
cn7_cycle[dev_cols].isnull().sum().sum()

In [ ]:
cn7_work = cn7_work.merge(
    cn7_cycle[
        ["TimeStamp"] + dev_cols
    ],
    on="TimeStamp",
    how="left"
)

In [ ]:
print("행 수:", len(cn7_work))
print("전체 컬럼 수:", cn7_work.shape[1])

print(
    "Deviation 결측:",
    cn7_work[dev_cols].isnull().sum().sum()
)

### 지금까지의 파생변수 의미

Injection_Time
→ 현재 공정값

Injection_Time_diff1
→ 바로 직전 Cycle보다 얼마나 변했나

Injection_Time_roll5_mean
→ 최근 5 Cycle의 평소 수준은 어땠나

Injection_Time_roll5_std
→ 최근 5 Cycle이 얼마나 불안정했나

Injection_Time_roll5_dev
→ 현재값이 최근 수준에서 얼마나 벗어났나

# 2. 최종 데이터 점검

### 최종 컬럼 및 로우 수 확인

In [ ]:
print("CN7 shape:", cn7_work.shape)
print("행 수:", len(cn7_work))
print("컬럼 수:", cn7_work.shape[1])

print("\n전체 결측치:")
print(cn7_work.isnull().sum().sum())

### 무한대 값 확인

In [ ]:
import numpy as np

num_cols = cn7_work.select_dtypes(include=np.number).columns

print(
    "inf 개수:",
    np.isinf(cn7_work[num_cols]).sum().sum()
)

### 모델에 넣지 않을 컬럼 제거

original_index   
Part_Name   
RH_LH   
Clamp_Open_Position

In [ ]:
cn7_work.columns.tolist()

# 3. x y group 분리

In [ ]:
TARGET = "PassOrFail"

groups_cn7 = cn7_work["TimeStamp"].copy()

X_cn7 = cn7_work.drop(
    columns=[TARGET, "TimeStamp"]
).copy()

y_cn7 = cn7_work[TARGET].copy()

In [ ]:
print("X shape      :", X_cn7.shape)
print("y shape      :", y_cn7.shape)
print("Group 개수   :", groups_cn7.nunique())

print("\nTarget 분포")
print(y_cn7.value_counts())

print("\nTarget 비율")
print(y_cn7.value_counts(normalize=True))

### 컬럼 타입 확인

In [ ]:
X_cn7.dtypes.value_counts()

In [ ]:
X_cn7.select_dtypes(
    exclude=np.number
).columns.tolist()

### 혹시 모르니까 데이터 저장

In [ ]:
cn7_work.to_csv(
    "train_cn7_preprocessed.csv",
    index=False,
    encoding="utf-8-sig"
)

### 4. train test 분리

StratifiedGroupKFold 

동일 TimeStamp의 LH/RH가 동일 생산 Cycle의 센서값을 공유하므로, 같은 Cycle이 학습셋과 검증셋에 동시에 포함되는 데이터 누수를 방지하기 위해 TimeStamp를 Group으로 설정하였다. 또한 정상 1,194건 대비 불량 17건으로 클래스 불균형이 매우 심하므로, Fold별 정상/불량 비율을 최대한 유지하기 위해 StratifiedGroupKFold를 사용하였다.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold


sgkf = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(X_cn7, y_cn7, groups=groups_cn7),
    start=1
):
    print(f"\n{'='*60}")
    print(f"Fold {fold}")
    print(f"{'='*60}")

    X_train_cn7 = X_cn7.iloc[train_idx]
    X_test_cn7  = X_cn7.iloc[test_idx]

    y_train_cn7 = y_cn7.iloc[train_idx]
    y_test_cn7  = y_cn7.iloc[test_idx]

    print("Train shape:", X_train_cn7.shape)
    print("Test shape :", X_test_cn7.shape)

    print(
        "Train target:",
        y_train_cn7.value_counts().to_dict()
    )

    print(
        "Test target:",
        y_test_cn7.value_counts().to_dict()
    )

    model_composition(
        X_train_cn7,
        X_test_cn7,
        y_train_cn7,
        y_test_cn7
    )

| 모델 | 4-Fold 평균 PR-AUC | 평균 Recall | 해석 |
|---|---:|---:|---|
| Logistic | 0.565 | 0.350 | baseline으로 괜찮음 |
| Random Forest | 0.602 | 0.250 | threshold에서 불량을 많이 놓침 |
| **LightGBM** | **0.822** | 0.300 | **현재 ranking 성능 가장 좋음** |
| XGBoost | 0.665 | 0.288 | 준수 |
| CatBoost | 0.659 | **0.363** | Recall은 상대적으로 좋음 |

PR-AUC : 실제 불량에게 정상보다 높은 확률을 잘 부여했는가  
recall : 확률 ranking은 상당히 괜찮은데, 0.5 threshold가 이 데이터에 맞지 않는다

CN7의 공정 변수에서 불량을 구분할 수 있는 신호가 확인되었고, 특히 LightGBM에서 높은 PR-AUC를 보였으나, 소수 불량 표본으로 Fold 간 변동성이 높아 추가 검증이 필요하다.

# 새로운 방식

### 새로운 featuer 세트 생성

In [ ]:
feature_sets_cn7 = {
    "V0_original": (
        SENSOR_COLS
        + ["Side_RH"]
    ),

    "V1_diff": (
        SENSOR_COLS
        + ["Side_RH"]
        + diff_cols
    ),

    "V2_roll_mean": (
        SENSOR_COLS
        + ["Side_RH"]
        + diff_cols
        + roll_mean_cols
    ),

    "V3_roll_std": (
        SENSOR_COLS
        + ["Side_RH"]
        + diff_cols
        + roll_mean_cols
        + roll_std_cols
    ),

    "V4_deviation": (
        SENSOR_COLS
        + ["Side_RH"]
        + diff_cols
        + roll_mean_cols
        + roll_std_cols
        + dev_cols
    )
}

In [ ]:
for name, cols in feature_sets_cn7.items():
    print(name, ":", len(cols))

### 같은 fold 고정

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

cn7_splits = list(
    sgkf.split(
        X_cn7,
        y_cn7,
        groups=groups_cn7
    )
)

### v0 ~ v4 전체 비교

In [ ]:
%%capture cn7_result

for version, feature_cols in feature_sets_cn7.items():

    print("\n")
    print("#" * 80)
    print(f"CN7 FEATURE SET : {version}")
    print(f"Feature Count   : {len(feature_cols)}")
    print("#" * 80)

    X_version = X_cn7[feature_cols]

    for fold, (train_idx, test_idx) in enumerate(
        cn7_splits,
        start=1
    ):
        print("\n")
        print("=" * 70)
        print(f"{version} | Fold {fold}")
        print("=" * 70)

        X_train_cn7 = X_version.iloc[train_idx]
        X_test_cn7  = X_version.iloc[test_idx]

        y_train_cn7 = y_cn7.iloc[train_idx]
        y_test_cn7  = y_cn7.iloc[test_idx]

        print(
            "Train:",
            X_train_cn7.shape,
            "| Target:",
            y_train_cn7.value_counts().to_dict()
        )

        print(
            "Test :",
            X_test_cn7.shape,
            "| Target:",
            y_test_cn7.value_counts().to_dict()
        )

        model_composition(
            X_train_cn7,
            X_test_cn7,
            y_train_cn7,
            y_test_cn7
        )

| Feature Set | 모델 | 평균 PR-AUC | 평균 Recall | 평균 F1 |
|---|---|---:|---:|---:|
| V0 Original 24개 | LightGBM | 0.535 | 0.288 | 0.392 |
| V1 + Diff 47개 | CatBoost | 0.477 | 0.225 | 0.310 |
| **V2 + Rolling Mean 70개** | **LightGBM** | **0.799** | **0.400** | **0.502** |
| V3 + Rolling Std 93개 | LightGBM | 0.803 | 0.363 | 0.481 |
| **V4 + Deviation 116개** | **LightGBM** | **0.823** | 0.300 | 0.450 |

V0 Original
PR-AUC 약 0.54

↓ diff 추가

V1
PR-AUC 약 0.37  ← 오히려 하락

↓ 최근 5 Cycle 평균 추가

V2
PR-AUC 약 0.80  ← 급상승

↓ 최근 5 Cycle 변동성 추가

V3
PR-AUC 약 0.80  ← 거의 유지

↓ 최근 평균 대비 편차 추가

V4
PR-AUC 약 0.82  ← 소폭 상승

V4가 불량 위험도를 순위화하는 능력은 가장 뛰어나지만, 기본 threshold 0.5에서는 불량 탐지 Recall이 낮다.

후보 1 — LightGBM + V4
- PR-AUC 가장 높음
- ranking 능력 가장 좋음
- threshold 조정 필요   
<br>

후보 2 — CatBoost + V2/V3
- PR-AUC는 LightGBM보다 낮음
- 기본 threshold 기준 Recall/F1 우수
- 비교 모델로 가치 있음|

직전 Cycle과의 단순 변화량은 독립적으로는 성능을 향상시키지 못했으며, 최근 5 Cycle의 평균 수준을 반영한 이동통계 특성에서 성능이 크게 향상되었다. 이후 Ablation Study를 통해 각 파생변수 군의 기여도를 검증하였다.

# diff1만 제거하고 비교

In [ ]:
feature_sets_ablation_cn7 = {
    # 원본 + Side + 최근 5 Cycle 평균
    "V5_no_diff_roll_mean": (
        SENSOR_COLS
        + ["Side_RH"]
        + roll_mean_cols
    ),

    # V5 + 최근 5 Cycle 표준편차
    "V6_no_diff_roll_std": (
        SENSOR_COLS
        + ["Side_RH"]
        + roll_mean_cols
        + roll_std_cols
    ),

    # V6 + 최근 평균 대비 편차
    "V7_no_diff_deviation": (
        SENSOR_COLS
        + ["Side_RH"]
        + roll_mean_cols
        + roll_std_cols
        + dev_cols
    )
}

for name, cols in feature_sets_ablation_cn7.items():
    print(name, ":", len(cols))

In [ ]:
for version, feature_cols in feature_sets_ablation_cn7.items():

    print("\n")
    print("#" * 80)
    print(f"CN7 ABLATION FEATURE SET : {version}")
    print(f"Feature Count            : {len(feature_cols)}")
    print("#" * 80)

    X_version = X_cn7[feature_cols]

    for fold, (train_idx, test_idx) in enumerate(
        cn7_splits,
        start=1
    ):
        print("\n")
        print("=" * 70)
        print(f"{version} | Fold {fold}")
        print("=" * 70)

        X_train_cn7 = X_version.iloc[train_idx]
        X_test_cn7  = X_version.iloc[test_idx]

        y_train_cn7 = y_cn7.iloc[train_idx]
        y_test_cn7  = y_cn7.iloc[test_idx]

        print(
            "Train:",
            X_train_cn7.shape,
            "| Target:",
            y_train_cn7.value_counts().to_dict()
        )

        print(
            "Test :",
            X_test_cn7.shape,
            "| Target:",
            y_test_cn7.value_counts().to_dict()
        )

        model_composition(
            X_train_cn7,
            X_test_cn7,
            y_train_cn7,
            y_test_cn7
        )

| Feature Set | LightGBM 평균 PR-AUC | 평균 Recall | 평균 F1 | Feature 수 |
|---|---:|---:|---:|---:|
| **V5: Original + Roll Mean** | **0.805** | 0.513 | **0.633** | **47** |
| V6: + Roll Std | 0.793 | **0.538** | 0.607 | 70 |
| V7: + Deviation | 0.787 | 0.425 | 0.564 | 93 |

복잡도를 116개에서 47개로 약 60% 줄이면서 PR-AUC 성능을 거의 유지하고, 실제 불량 탐지 Recall과 F1은 개선하였다.  

-> diff를 제거했는데 오히려 성능이 소폭 개선됨

# CN7 최종 feature

In [ ]:
features_cn7_final = (
    SENSOR_COLS
    + ["Side_RH"]
    + roll_mean_cols
)

print("CN7 최종 Feature 수:", len(features_cn7_final))

Feature Ablation 결과, 직전 1 Cycle 변화량(diff1)은 성능 향상에 기여하지 않았으며, 최근 5 Cycle의 평균 공정 상태(Rolling Mean)를 추가했을 때 PR-AUC가 크게 향상되었다. 반면 Rolling Std와 Deviation을 추가했을 때 모델 복잡도 대비 추가 성능 향상이 제한적이어서 최종적으로 원본 공정 변수 + Side + 최근 5 Cycle 평균으로 구성된 47개 변수를 채택하였다.

# Threshold 조정

In [ ]:
features_cn7_final = (
    SENSOR_COLS
    + ["Side_RH"]
    + roll_mean_cols
)

X_cn7_final = X_cn7[features_cn7_final].copy()

print("최종 Feature 수:", X_cn7_final.shape[1])

In [ ]:
import numpy as np

from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score

oof_proba_cn7 = np.zeros(len(X_cn7_final))

for fold, (train_idx, val_idx) in enumerate(
    cn7_splits,
    start=1
):
    X_train = X_cn7_final.iloc[train_idx]
    X_val   = X_cn7_final.iloc[val_idx]

    y_train = y_cn7.iloc[train_idx]
    y_val   = y_cn7.iloc[val_idx]

    model = LGBMClassifier(
        random_state=42,
        verbose=-1
    )

    model.fit(
        X_train,
        y_train
    )

    val_proba = model.predict_proba(
        X_val
    )[:, 1]

    oof_proba_cn7[val_idx] = val_proba

    fold_pr_auc = average_precision_score(
        y_val,
        val_proba
    )

    print(
        f"Fold {fold} PR-AUC: "
        f"{fold_pr_auc:.4f}"
    )

In [ ]:
oof_pr_auc = average_precision_score(
    y_cn7,
    oof_proba_cn7
)

print(
    "OOF PR-AUC:",
    round(oof_pr_auc, 4)
)

In [ ]:
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

threshold_results = []

thresholds = np.arange(
    0.05,
    0.96,
    0.05
)

for threshold in thresholds:

    pred = (
        oof_proba_cn7 >= threshold
    ).astype(int)

    precision = precision_score(
        y_cn7,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_cn7,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_cn7,
        pred,
        zero_division=0
    )

    f2 = fbeta_score(
        y_cn7,
        pred,
        beta=2,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_cn7,
        pred
    ).ravel()

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "F2": f2,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df

In [ ]:
best_f1 = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

best_f1

In [ ]:
best_f2 = threshold_df.loc[
    threshold_df["F2"].idxmax()
]

best_f2

In [ ]:
threshold_df[
    threshold_df["Recall"] >= 0.8
].sort_values(
    "Precision",
    ascending=False
)

In [ ]:
threshold_df.round({
    "Threshold": 2,
    "Precision": 4,
    "Recall": 4,
    "F1": 4,
    "F2": 4
})

최적의 threshold = 0.05   

불량의 76.5%를 검출하면서 정상 제품 1,194개 중 단 3개만 오탐

| Threshold | Precision | Recall | F1 | F2 | TP | FP | FN |
|---:|---:|---:|---:|---:|---:|---:|---:|
| **0.05** | **0.8125** | **0.7647** | **0.7879** | **0.7738** | **13** | **3** | **4** |
| 0.10 | 0.8571 | 0.7059 | 0.7742 | 0.7317 | 12 | 2 | 5 |
| 0.30 | 0.9000 | 0.5294 | 0.6667 | 0.5769 | 9 | 1 | 8 |
| 0.50 | 0.9000 | 0.5294 | 0.6667 | 0.5769 | 9 | 1 | 8 |
| 0.55 | 1.0000 | 0.5294 | 0.6923 | 0.5844 | 9 | 0 | 8 |

### 좀 더 자세히 탐지

In [ ]:
threshold_results_fine = []

thresholds_fine = np.arange(
    0.005,
    0.101,
    0.005
)

for threshold in thresholds_fine:

    pred = (
        oof_proba_cn7 >= threshold
    ).astype(int)

    precision = precision_score(
        y_cn7,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_cn7,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_cn7,
        pred,
        zero_division=0
    )

    f2 = fbeta_score(
        y_cn7,
        pred,
        beta=2,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_cn7,
        pred
    ).ravel()

    threshold_results_fine.append({
        "Threshold": round(threshold, 3),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "F2": f2,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })


threshold_fine_df = pd.DataFrame(
    threshold_results_fine
)

threshold_fine_df.round(4)

In [ ]:
threshold_fine_df[
    threshold_fine_df["Recall"] >= 0.8
].sort_values(
    ["Precision", "F2"],
    ascending=False
)

기본 Threshold 0.5에서는 불량 Recall이 52.9%에 그쳤으나, OOF 예측 확률을 활용한 Threshold 조정을 통해 0.05에서 Recall 76.5%, Precision 81.3%, F1 78.8%를 달성하였다. 정상 1,194건 중 오탐은 3건으로, 낮은 오탐 비용으로 불량 탐지율을 크게 개선하였다.

기본 임계값 0.5에서는 불량 Recall이 52.9%였으나, OOF 예측 확률 기반 임계값 최적화를 통해 threshold 0.035에서 Precision 82.35%, Recall 82.35%, F1 82.35%를 확보하였다. 특히 정상 1,194건 중 오탐은 3건에 불과하면서 17건의 불량 중 14건을 탐지하였다.

# StratifiedGroupKFold random seed

In [ ]:
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

# ==========================================
# CN7 반복 검증 설정
# ==========================================

SEEDS = [42, 100, 2026, 777, 1234]

N_SPLITS = 4
FIXED_THRESHOLD = 0.035

X_final = X_cn7[features_cn7_final].copy()
y_final = y_cn7.copy()
groups_final = groups_cn7.copy()

seed_results = []

In [ ]:
for seed in SEEDS:

    sgkf = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    oof_proba = np.zeros(len(X_final))

    print("\n" + "=" * 70)
    print(f"SEED : {seed}")
    print("=" * 70)

    for fold, (train_idx, val_idx) in enumerate(
        sgkf.split(
            X_final,
            y_final,
            groups=groups_final
        ),
        start=1
    ):

        X_train = X_final.iloc[train_idx]
        X_val   = X_final.iloc[val_idx]

        y_train = y_final.iloc[train_idx]
        y_val   = y_final.iloc[val_idx]

        # 앞에서 사용한 LightGBM과 동일한 파라미터 사용
        model = LGBMClassifier(
            random_state=42,
            verbose=-1
        )

        model.fit(
            X_train,
            y_train
        )

        val_proba = model.predict_proba(
            X_val
        )[:, 1]

        oof_proba[val_idx] = val_proba

        fold_pr_auc = average_precision_score(
            y_val,
            val_proba
        )

        print(
            f"Fold {fold} "
            f"| Defect: {y_val.sum()} "
            f"| PR-AUC: {fold_pr_auc:.4f}"
        )

    # ==========================================
    # Seed 전체 OOF 평가
    # ==========================================

    pred = (
        oof_proba >= FIXED_THRESHOLD
    ).astype(int)

    pr_auc = average_precision_score(
        y_final,
        oof_proba
    )

    precision = precision_score(
        y_final,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_final,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_final,
        pred,
        zero_division=0
    )

    f2 = fbeta_score(
        y_final,
        pred,
        beta=2,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_final,
        pred
    ).ravel()

    seed_results.append({
        "Seed": seed,
        "PR_AUC": pr_auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "F2": f2,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

In [ ]:
seed_result_df = pd.DataFrame(
    seed_results
)

seed_result_df.round(4)

In [ ]:
summary_cn7 = (
    seed_result_df[
        [
            "PR_AUC",
            "Precision",
            "Recall",
            "F1",
            "F2"
        ]
    ]
    .agg(["mean", "std", "min", "max"])
    .T
)

summary_cn7.round(4)

In [ ]:
seed_result_df[
    [
        "Seed",
        "Precision",
        "Recall",
        "F1",
        "F2",
        "TP",
        "FP",
        "FN"
    ]
].round(4)

| 지표 | 평균 | 표준편차 | 최소 | 최대 |
|---|---:|---:|---:|---:|
| PR-AUC | **0.7613** | 0.0529 | 0.6976 | 0.8372 |
| Precision | **0.6950** | 0.0737 | 0.6471 | 0.8235 |
| Recall | **0.7176** | 0.0767 | 0.6471 | 0.8235 |
| F1 | **0.7051** | 0.0694 | 0.6471 | 0.8235 |
| F2 | **0.7123** | 0.0722 | 0.6471 | 0.8235 |

# threshold를 여러 seed에서 동시에 최적화

### seed별 oof 저장

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score

SEEDS = [42, 100, 2026, 777, 1234]

oof_by_seed = {}

for seed in SEEDS:

    sgkf = StratifiedGroupKFold(
        n_splits=4,
        shuffle=True,
        random_state=seed
    )

    oof_proba = np.zeros(len(X_cn7_final))

    for train_idx, val_idx in sgkf.split(
        X_cn7_final,
        y_cn7,
        groups=groups_cn7
    ):
        X_train = X_cn7_final.iloc[train_idx]
        X_val   = X_cn7_final.iloc[val_idx]

        y_train = y_cn7.iloc[train_idx]

        model = LGBMClassifier(
            random_state=42,
            verbose=-1
        )

        model.fit(
            X_train,
            y_train
        )

        oof_proba[val_idx] = model.predict_proba(
            X_val
        )[:, 1]

    oof_by_seed[seed] = oof_proba

### threshold 별로 5개 seed 평균 계싼

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

threshold_summary = []

thresholds = np.arange(
    0.005,
    0.101,
    0.005
)

for threshold in thresholds:

    seed_metrics = []

    for seed, proba in oof_by_seed.items():

        pred = (
            proba >= threshold
        ).astype(int)

        precision = precision_score(
            y_cn7,
            pred,
            zero_division=0
        )

        recall = recall_score(
            y_cn7,
            pred,
            zero_division=0
        )

        f1 = f1_score(
            y_cn7,
            pred,
            zero_division=0
        )

        f2 = fbeta_score(
            y_cn7,
            pred,
            beta=2,
            zero_division=0
        )

        tn, fp, fn, tp = confusion_matrix(
            y_cn7,
            pred
        ).ravel()

        seed_metrics.append({
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "F2": f2,
            "TP": tp,
            "FP": fp,
            "FN": fn
        })

    temp = pd.DataFrame(seed_metrics)

    threshold_summary.append({
        "Threshold": round(threshold, 3),

        "Precision_mean": temp["Precision"].mean(),
        "Precision_std": temp["Precision"].std(),

        "Recall_mean": temp["Recall"].mean(),
        "Recall_std": temp["Recall"].std(),

        "F1_mean": temp["F1"].mean(),
        "F1_std": temp["F1"].std(),

        "F2_mean": temp["F2"].mean(),
        "F2_std": temp["F2"].std(),

        "TP_mean": temp["TP"].mean(),
        "FP_mean": temp["FP"].mean(),
        "FN_mean": temp["FN"].mean()
    })

threshold_seed_df = pd.DataFrame(
    threshold_summary
)

threshold_seed_df.round(4)

### 평균 F1 최고 threshold

In [ ]:
threshold_seed_df.loc[
    threshold_seed_df["F1_mean"].idxmax()
]

### 평균 F2 최고 threshold

In [ ]:
threshold_seed_df.loc[
    threshold_seed_df["F2_mean"].idxmax()
]

In [ ]:
threshold_seed_df.sort_values(
    "F2_mean",
    ascending=False
).head(10).round(4)

| Threshold | Precision | Recall | F1 | F2 | 평균 TP | 평균 FP |
|---:|---:|---:|---:|---:|---:|---:|
| 0.010 | 0.6323 | **0.7529** | 0.6861 | **0.7243** | 12.8 | 7.4 |
| **0.020** | **0.6810** | **0.7294** | **0.7033** | **0.7185** | 12.4 | 5.8 |
| 0.035 | **0.6950** | 0.7176 | **0.7051** | 0.7123 | 12.2 | 5.4 |

순수 F1 최적화 → 0.035   
불량 미탐 최소화(F2) → 0.010   
실무적인 Precision–Recall 절충 → 0.020

5개 random seed 기반 반복 StratifiedGroupKFold 검증 결과, F1 기준 최적 임계값은 0.035였으나, 제조 불량의 미탐 비용을 고려하여 Recall과 F2를 추가 검토하였다. Threshold 0.020에서 평균 Precision 68.1%, Recall 72.9%, F1 70.3%, F2 71.9%를 확보하여 탐지율과 오탐 간 균형이 우수한 운영 임계값으로 선정하였다.

=======================================================================================
# 지금까지의 결론

### 1. 데이터 개요

| 항목 | 내용 |
|---|---:|
| 데이터셋 | CN7 사출성형 데이터 |
| 전체 데이터 | 1,211행 |
| 정상 `PassOrFail=0` | 1,194건 |
| 불량 `PassOrFail=1` | 17건 |
| 불량 비율 | 약 1.40% |
| 고유 생산 Cycle (`TimeStamp`) | 606개 |
| Target | `PassOrFail` |
| 정상 | 0 |
| 불량 | 1 |

### 2. 기본 전처리

| 처리 항목 | 처리 방법 | 이유 |
|---|---|---|
| `TimeStamp` | `datetime` 변환 | 시간 순서 및 Cycle 그룹 생성 |
| 정렬 | `TimeStamp`, `RH_LH` 기준 정렬 | 생산 순서 유지 |
| `original_index` | 제거 | 원본 행 번호로 모델 학습할 필요 없음 |
| `Clamp_Open_Position` | 제거 | 모든 값이 동일한 상수 컬럼 |
| `PART_NAME` | 제거 | `RH_LH`와 사실상 중복 정보 |
| `RH_LH` | `Side_RH`로 변환 | LH=0, RH=1 |
| 결측치 | 별도 대체 없음 | 원본 기준 결측치 없음 |
| Scaling | 추가 적용하지 않음 | 기존 센서 데이터가 Standard Scaling된 상태 |
| 이상치 제거 | 수행하지 않음 | 극단값 자체가 불량 신호일 가능성이 있음 |
| 저분산 제거 | 일괄 제거하지 않음 | 미세한 변화가 불량과 관련될 수 있음 |

### 3. 검증 방법

StratifiedGroupKFold 적용
> 동일 TimeStamp의 LH/RH는 동일 생산 Cycle의 공정 센서값을 공유하기 때문에, 일반적인 train_test_split()을 사용할 경우 동일한 공정 Cycle이 Train과 Validation에 동시에 포함되어 **데이터 누수(Data Leakage)**가 발생할 가능성이 있다.
따라서 동일 생산 Cycle의 LH/RH가 반드시 같은 Fold에 포함되도록 TimeStamp를 Group으로 설정하였다. 동시에 불량 데이터가 17건뿐이므로 Fold별 정상/불량 비율을 최대한 유지하기 위해 StratifiedGroupKFold를 사용하였다.
동일 생산 Cycle의 데이터 누수를 방지하면서 클래스 불균형을 고려하기 위해 StratifiedGroupKFold를 적용하였다.

### 4. cycle 기반 feature engineering

동일 TimeStamp의 LH/RH 센서값이 동일하므로, 파생변수는 행 단위가 아니라 생산 Cycle 단위에서 생성한 후 LH/RH 데이터에 다시 병합하였다.
생산 중 장시간 중단 전후의 데이터를 직전 Cycle로 연결하지 않기 위해 Cycle 간 시간 간격이 300초(5분) 이상인 경우 새로운 생산 세션으로 구분하였다.

### 5. 생성한 파생 변수

| 파생변수 | 생성 방식 | 의미 | 최종 사용 여부 |
|---|---|---|---|
| `Side_RH` | `RH=1`, `LH=0` | 동일 공정조건에서도 RH/LH에 따라 품질 결과가 달라질 수 있으므로 제품 Side 정보 반영 | **사용** |
| `diff1` | `현재 Cycle - 직전 Cycle` | 바로 직전 생산 Cycle 대비 센서값 변화량 | 제외 |
| `roll5_mean` | 현재 Cycle을 제외한 직전 5개 Cycle 평균 | 최근 공정의 기준 수준 및 공정 상태 변화 반영 | **사용** |
| `roll5_std` | 현재 Cycle을 제외한 직전 5개 Cycle 표준편차 | 최근 공정의 변동성 및 불안정성 표현 | 제외 |
| `roll5_dev` | `현재값 - roll5_mean` | 현재 Cycle이 최근 공정 수준에서 얼마나 벗어났는지 표현 | 제외 |

### 6. feature set 구성

파생변수를 단계적으로 추가함

| Version | Feature 구성 | Feature 수 |
|---|---|---:|
| V0 | Original + Side | 24 |
| V1 | Original + Side + Diff | 47 |
| V2 | Original + Side + Diff + Rolling Mean | 70 |
| V3 | V2 + Rolling Std | 93 |
| V4 | V3 + Deviation | 116 |
| V5 | Original + Side + Rolling Mean | **47** |
| V6 | V5 + Rolling Std | 70 |
| V7 | V6 + Deviation | 93 |

### 7. 1차 feature 실험 결과

| Feature Set | Feature 수 | LightGBM 평균 PR-AUC |
|---|---:|---:|
| V0 Original | 24 | 0.535 |
| V1 + Diff | 47 | 0.369 |
| V2 + Rolling Mean | 70 | 0.799 |
| V3 + Rolling Std | 93 | 0.803 |
| **V4 + Deviation** | 116 | **0.822** |

단순 직전 Cycle 변화량인 diff1을 추가한 V1은 오히려 성능이 하락하였다.
반면 roll5_mean을 추가한 V2부터 PR-AUC가 크게 상승하였다.
따라서 CN7 불량은 단순히 직전 Cycle과의 순간적인 변화보다는 최근 여러 Cycle 동안 형성된 공정 수준과 더 관련이 있을 가능성이 높다고 판단하였다.

### 8. ablation study

| Version | 구성 |
|---|---|
| V5 | Original + Side + Rolling Mean |
| V6 | V5 + Rolling Std |
| V7 | V6 + Deviation |


lightgbm 결과
| Feature Set | Feature 수 | 평균 PR-AUC | 평균 Recall | 평균 F1 |
|---|---:|---:|---:|---:|
| **V5** | **47** | **0.805** | 0.513 | **0.633** |
| V6 | 70 | 0.793 | **0.538** | 0.607 |
| V7 | 93 | 0.787 | 0.425 | 0.564 |
| V4 | 116 | **0.822** | 0.300 | 0.450 |

### 최종 feature set 선정

> Original Sensor 23개    
> +   
> Side_RH 1개   
> +   
> Roll5 Mean 23개   
> ----------------   
> 총 47개 Feature

V4가 PR-AUC만 보면 약 0.822로 가장 높았지만 116개의 Feature를 사용한다.
반면 V5는 47개의 Feature만 사용하면서 PR-AUC 약 0.805를 유지하였고, Recall 및 F1은 오히려 더 우수하였다.
따라서 Feature 수를 약 60% 줄이면서 성능을 거의 유지할 수 있는 V5가 모델 복잡도, 일반화 가능성, 해석 가능성 측면에서 가장 효율적인 Feature Set으로 판단하였다.
또한 diff1을 제거한 V5가 V2보다 성능이 소폭 개선되어, 직전 1 Cycle의 변화량은 CN7 불량 예측에 큰 도움이 되지 않는 것으로 판단하였다.

### 10. 모델 비교

| Model | 평균 PR-AUC | 평균 Recall | 평균 F1 |
|---|---:|---:|---:|
| Logistic Regression | 약 0.630 | 0.350 | 0.435 |
| Random Forest | 약 0.664 | 0.350 | 0.476 |
| **LightGBM** | **약 0.805** | 0.513 | **0.633** |
| XGBoost | 약 0.660 | 0.413 | 0.500 |
| CatBoost | 약 0.695 | **0.525** | 0.616 |

### 11. 현재 선택한 모델 > LightGBM

| Parameter | 값 | 설명 |
|---|---:|---|
| Model | LightGBM | 최종 후보 모델 |
| `random_state` | 42 | 재현성 확보 |
| `verbose` | -1 | 불필요한 학습 Warning 출력 억제 |
| Class Weight | 미적용 | 추후 실험 예정 |
| SMOTE | 미적용 | 추후 실험 예정 |
| RandomOverSampler | 미적용 | 추후 실험 예정 |
| Hyperparameter Tuning | 미실시 | 추후 진행 예정 |

### 12. Threshold Optimization

| Threshold | Precision | Recall | F1 | F2 | TP | FP | FN |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 0.035 | **0.8235** | **0.8235** | **0.8235** | **0.8235** | 14 | 3 | 3 |
| 0.050 | 0.8125 | 0.7647 | 0.7879 | 0.7738 | 13 | 3 | 4 |
| 0.100 | 0.8571 | 0.7059 | 0.7742 | 0.7317 | 12 | 2 | 5 |
| 0.500 | 0.9000 | 0.5294 | 0.6667 | 0.5769 | 9 | 1 | 8 |

### 13. 반복 StratifiedGroupKFold 검증

| 지표 | Mean | Std | Min | Max |
|---|---:|---:|---:|---:|
| PR-AUC | **0.7613** | 0.0529 | 0.6976 | 0.8372 |
| Precision | 0.6950 | 0.0737 | 0.6471 | 0.8235 |
| Recall | 0.7176 | 0.0767 | 0.6471 | 0.8235 |
| F1 | 0.7051 | 0.0694 | 0.6471 | 0.8235 |
| F2 | 0.7123 | 0.0722 | 0.6471 | 0.8235 |

seed별 threshold 0.035 성능
| Seed | Precision | Recall | F1 | F2 | TP | FP | FN |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 42 | 0.8235 | 0.8235 | 0.8235 | 0.8235 | 14 | 3 | 3 |
| 100 | 0.6875 | 0.6471 | 0.6667 | 0.6548 | 11 | 5 | 6 |
| 2026 | 0.6667 | 0.7059 | 0.6857 | 0.6977 | 12 | 6 | 5 |
| 777 | 0.6500 | 0.7647 | 0.7027 | 0.7386 | 13 | 7 | 4 |
| 1234 | 0.6471 | 0.6471 | 0.6471 | 0.6471 | 11 | 6 | 6 |

### 14. Robust Threshold 탐색

| Threshold | Precision Mean | Recall Mean | F1 Mean | F2 Mean | TP Mean | FP Mean | FN Mean |
|---:|---:|---:|---:|---:|---:|---:|---:|
| **0.010** | 0.6323 | **0.7529** | 0.6861 | **0.7243** | 12.8 | 7.4 | 4.2 |
| 0.015 | 0.6552 | 0.7412 | 0.6938 | 0.7209 | 12.6 | 6.6 | 4.4 |
| **0.020** | **0.6810** | **0.7294** | **0.7033** | **0.7185** | 12.4 | 5.8 | 4.6 |
| 0.030 | 0.6858 | 0.7176 | 0.7004 | 0.7104 | 12.2 | 5.6 | 4.8 |
| **0.035** | **0.6950** | 0.7176 | **0.7051** | 0.7123 | 12.2 | 5.4 | 4.8 |
| 0.050 | 0.7038 | 0.6941 | 0.6981 | 0.6955 | 11.8 | 5.0 | 5.2 |
| 0.055 | 0.7129 | 0.6941 | 0.7023 | 0.6971 | 11.8 | 4.8 | 5.2 |

### 15. Threshold 기준 정리

| 목적 | Threshold | 결과 |
|---|---:|---|
| **F1 최대** | **0.035** | F1 ≈ 0.705 |
| **F2 최대 / Recall 우선** | **0.010** | F2 ≈ 0.724, Recall ≈ 0.753 |
| **Precision–Recall 절충** | **0.020** | Precision ≈ 0.681, Recall ≈ 0.729, F1 ≈ 0.703 |

### 16. 현재 CN7 최종 구성

| 항목 | 현재 선택 |
|---|---|
| 데이터 | CN7 |
| Target | `PassOrFail` |
| Validation | `StratifiedGroupKFold` |
| Group | `TimeStamp` |
| Fold 수 | 4 |
| 반복 Seed | 42, 100, 2026, 777, 1234 |
| Original Sensor | 23개 |
| Side Feature | `Side_RH` |
| Temporal Feature | `roll5_mean` |
| Rolling Window | 5 Cycle |
| 현재 Cycle 포함 여부 | 제외 (`shift(1)`) |
| 최종 Feature 수 | **47개** |
| 제거 Feature | `diff1`, `roll5_std`, `roll5_dev` |
| 최종 모델 후보 | **LightGBM** |
| `random_state` | 42 |
| `verbose` | -1 |
| Primary Metric | **PR-AUC** |
| Secondary Metric | Precision, Recall, F1, F2 |
| F1 최적 Threshold | 0.035 |
| F2 최적 Threshold | 0.010 |
| 운영 Threshold 후보 | **0.020** |
| Class Weight | 아직 미실험 |
| ROS | 아직 미실험 |
| SMOTE | 아직 미실험 |
| Hyperparameter Tuning | 아직 미실험 |

### 17. 현재까지 최고 성능

단일분할 기준 최고 threshold 성능   
-> lightgbm + v5에서 seed 42, oof prediction 기준

| Threshold | Precision | Recall | F1 | F2 | TP | FP | FN | TN |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| **0.035** | **0.8235** | **0.8235** | **0.8235** | **0.8235** | **14** | **3** | **3** | **1191** |

반복 group cv 기준 성능
| Metric | Mean | Std | Min | Max |
|---|---:|---:|---:|---:|
| **PR-AUC** | **0.7613** | 0.0529 | 0.6976 | 0.8372 |
| Precision | 0.6950 | 0.0737 | 0.6471 | 0.8235 |
| Recall | 0.7176 | 0.0767 | 0.6471 | 0.8235 |
| F1 | **0.7051** | 0.0694 | 0.6471 | 0.8235 |
| F2 | **0.7123** | 0.0722 | 0.6471 | 0.8235 |

threshold 별 최고 결과
| 기준 | Threshold | Precision | Recall | F1 | F2 |
|---|---:|---:|---:|---:|---:|
| **F1 최대** | **0.035** | 0.6950 | 0.7176 | **0.7051** | 0.7123 |
| **F2 최대** | **0.010** | 0.6323 | **0.7529** | 0.6861 | **0.7243** |
| **운영 절충안** | **0.020** | **0.6810** | **0.7294** | **0.7033** | **0.7185** |

CN7에서는 단일 Cycle의 순간 변화보다 최근 5 Cycle의 평균적인 공정 상태가 불량 예측에 가장 효과적이었으며, 최종적으로 Original Sensor + Side + Roll5 Mean으로 구성된 47개 Feature와 LightGBM을 선정하였다. 반복 Group CV 기준 PR-AUC 0.7613, F1 0.7051을 기록했으며, 단일 OOF 실험에서는 Precision·Recall·F1 모두 0.8235까지 확인되었다.

# 결론 : 지금까지 GOAT 점수

| Model | Feature Set | Feature 수 | Threshold | Precision | Recall | F1 | PR-AUC |
|---|---|---:|---:|---:|---:|---:|---:|
| **LightGBM** | **Original + Side + Roll5 Mean (V5)** | **47** | **0.035** | **0.8235** | **0.8235** | **0.8235** | **0.8050*** |

얘는 반복 검증 기준 대표 성능

| Model | Threshold | PR-AUC | Precision | Recall | F1 | F2 |
|---|---:|---:|---:|---:|---:|---:|
| **LightGBM** | **0.035** | **0.7613 ± 0.0529** | **0.6950 ± 0.0737** | **0.7176 ± 0.0767** | **0.7051 ± 0.0694** | **0.7123 ± 0.0722** |

# 불균형 처리 방법 준비

In [ ]:
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

from imblearn.over_sampling import (
    RandomOverSampler,
    SMOTE
)

In [ ]:
X_final = X_cn7[
    features_cn7_final
].copy()

y_final = y_cn7.copy()

groups_final = groups_cn7.copy()

print(X_final.shape)
print(y_final.value_counts())

In [ ]:
METHODS = [
    "Baseline",
    "ClassWeight",
    "RandomOverSampler",
    "SMOTE"
]

SEEDS = [
    42,
    100,
    2026,
    777,
    1234
]

N_SPLITS = 4

반복 Group CV + OOF 생성

In [ ]:
oof_results = {}

for method in METHODS:

    print("\n" + "#" * 80)
    print("METHOD :", method)
    print("#" * 80)

    oof_results[method] = {}

    for seed in SEEDS:

        sgkf = StratifiedGroupKFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=seed
        )

        oof_proba = np.zeros(
            len(X_final)
        )

        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(
            sgkf.split(
                X_final,
                y_final,
                groups=groups_final
            ),
            start=1
        ):

            X_train = X_final.iloc[
                train_idx
            ].copy()

            X_val = X_final.iloc[
                val_idx
            ].copy()

            y_train = y_final.iloc[
                train_idx
            ].copy()

            y_val = y_final.iloc[
                val_idx
            ].copy()


            # =============================
            # 1. Baseline
            # =============================

            if method == "Baseline":

                model = LGBMClassifier(
                    random_state=42,
                    verbose=-1
                )


            # =============================
            # 2. Class Weight
            # =============================

            elif method == "ClassWeight":

                model = LGBMClassifier(
                    class_weight="balanced",
                    random_state=42,
                    verbose=-1
                )


            # =============================
            # 3. RandomOverSampler
            # =============================

            elif method == "RandomOverSampler":

                sampler = RandomOverSampler(
                    random_state=seed
                )

                X_train, y_train = (
                    sampler.fit_resample(
                        X_train,
                        y_train
                    )
                )

                model = LGBMClassifier(
                    random_state=42,
                    verbose=-1
                )


            # =============================
            # 4. SMOTE
            # =============================

            elif method == "SMOTE":

                sampler = SMOTE(
                    random_state=seed,
                    k_neighbors=3
                )

                X_train, y_train = (
                    sampler.fit_resample(
                        X_train,
                        y_train
                    )
                )

                model = LGBMClassifier(
                    random_state=42,
                    verbose=-1
                )


            # 모델 학습
            model.fit(
                X_train,
                y_train
            )


            # Validation probability
            val_proba = (
                model
                .predict_proba(X_val)[:, 1]
            )

            oof_proba[
                val_idx
            ] = val_proba


            fold_pr_auc = (
                average_precision_score(
                    y_val,
                    val_proba
                )
            )

            fold_scores.append(
                fold_pr_auc
            )


        # Seed별 OOF 저장
        oof_results[
            method
        ][seed] = oof_proba


        seed_pr_auc = (
            average_precision_score(
                y_final,
                oof_proba
            )
        )

        print(
            f"{seed} | "
            f"PR-AUC = {seed_pr_auc:.4f}"
        )

PR-ACU 비교

In [ ]:
pr_auc_results = []

for method in METHODS:

    for seed in SEEDS:

        proba = (
            oof_results[
                method
            ][seed]
        )

        score = (
            average_precision_score(
                y_final,
                proba
            )
        )

        pr_auc_results.append({
            "Method": method,
            "Seed": seed,
            "PR_AUC": score
        })


pr_auc_df = pd.DataFrame(
    pr_auc_results
)

pr_auc_summary = (
    pr_auc_df
    .groupby("Method")["PR_AUC"]
    .agg([
        "mean",
        "std",
        "min",
        "max"
    ])
    .sort_values(
        "mean",
        ascending=False
    )
)

pr_auc_summary.round(4)

Threshold 최적화

In [ ]:
thresholds = np.arange(
    0.005,
    0.501,
    0.005
)

threshold_all = []

for method in METHODS:

    for threshold in thresholds:

        seed_metrics = []

        for seed in SEEDS:

            proba = (
                oof_results[
                    method
                ][seed]
            )

            pred = (
                proba >= threshold
            ).astype(int)

            precision = (
                precision_score(
                    y_final,
                    pred,
                    zero_division=0
                )
            )

            recall = (
                recall_score(
                    y_final,
                    pred,
                    zero_division=0
                )
            )

            f1 = (
                f1_score(
                    y_final,
                    pred,
                    zero_division=0
                )
            )

            f2 = (
                fbeta_score(
                    y_final,
                    pred,
                    beta=2,
                    zero_division=0
                )
            )

            tn, fp, fn, tp = (
                confusion_matrix(
                    y_final,
                    pred
                ).ravel()
            )

            seed_metrics.append({
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
                "F2": f2,
                "TP": tp,
                "FP": fp,
                "FN": fn,
                "TN": tn
            })


        temp = pd.DataFrame(
            seed_metrics
        )

        threshold_all.append({

            "Method":
                method,

            "Threshold":
                round(
                    threshold,
                    3
                ),

            "Precision_mean":
                temp[
                    "Precision"
                ].mean(),

            "Recall_mean":
                temp[
                    "Recall"
                ].mean(),

            "F1_mean":
                temp[
                    "F1"
                ].mean(),

            "F2_mean":
                temp[
                    "F2"
                ].mean(),

            "TP_mean":
                temp[
                    "TP"
                ].mean(),

            "FP_mean":
                temp[
                    "FP"
                ].mean(),

            "FN_mean":
                temp[
                    "FN"
                ].mean()
        })


imbalance_threshold_df = (
    pd.DataFrame(
        threshold_all
    )
)

각 방법에서 F1 최고값 추출

In [ ]:
best_f1_by_method = (
    imbalance_threshold_df
    .loc[
        imbalance_threshold_df
        .groupby("Method")[
            "F1_mean"
        ]
        .idxmax()
    ]
    .sort_values(
        "F1_mean",
        ascending=False
    )
)

best_f1_by_method.round(4)

In [ ]:
pr_auc_summary.round(4)

In [ ]:
best_f1_by_method.round(4)

# 튜닝 파라미터 후보

In [ ]:
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier

import numpy as np
import pandas as pd

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],

    "num_leaves": [7, 15, 31],
    "max_depth": [3, 5, 7, -1],

    "min_child_samples": [5, 10, 20, 30, 50],

    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],

    "reg_alpha": [0, 0.1, 0.5, 1.0],
    "reg_lambda": [0, 0.1, 0.5, 1.0]
}

### random search 후보 생성

In [ ]:
param_list = list(
    ParameterSampler(
        param_grid,
        n_iter=30,
        random_state=42
    )
)

print("실험할 파라미터 조합:", len(param_list))

### 반복 stratified group k fold 기준 튜닝

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

SEEDS = [42, 100, 2026, 777, 1234]

X_tune = X_cn7[features_cn7_final].copy()
y_tune = y_cn7.copy()
groups_tune = groups_cn7.copy()

tuning_results = []

In [ ]:
for trial, params in enumerate(param_list, start=1):

    seed_scores = []

    for seed in SEEDS:

        sgkf = StratifiedGroupKFold(
            n_splits=4,
            shuffle=True,
            random_state=seed
        )

        oof_proba = np.zeros(
            len(X_tune)
        )

        for train_idx, val_idx in sgkf.split(
            X_tune,
            y_tune,
            groups=groups_tune
        ):

            X_train = X_tune.iloc[train_idx]
            X_val   = X_tune.iloc[val_idx]

            y_train = y_tune.iloc[train_idx]
            y_val   = y_tune.iloc[val_idx]

            model = LGBMClassifier(
                **params,

                class_weight="balanced",

                random_state=42,
                verbose=-1
            )

            model.fit(
                X_train,
                y_train
            )

            oof_proba[val_idx] = (
                model.predict_proba(
                    X_val
                )[:, 1]
            )

        seed_pr_auc = (
            average_precision_score(
                y_tune,
                oof_proba
            )
        )

        seed_scores.append(
            seed_pr_auc
        )

    tuning_results.append({
        "Trial": trial,
        "PR_AUC_mean": np.mean(seed_scores),
        "PR_AUC_std": np.std(seed_scores),

        **params
    })

    print(
        f"Trial {trial:02d} | "
        f"PR-AUC = {np.mean(seed_scores):.4f} "
        f"± {np.std(seed_scores):.4f}"
    )

### 결과 정리

In [ ]:
tuning_df = pd.DataFrame(
    tuning_results
)

tuning_df = (
    tuning_df
    .sort_values(
        "PR_AUC_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

tuning_df.head(10)

### 최고 파라미터 확인

In [ ]:
best_result = tuning_df.iloc[0]

best_result

In [ ]:
best_params = {
    "n_estimators":
        int(best_result["n_estimators"]),

    "learning_rate":
        best_result["learning_rate"],

    "num_leaves":
        int(best_result["num_leaves"]),

    "max_depth":
        int(best_result["max_depth"]),

    "min_child_samples":
        int(best_result["min_child_samples"]),

    "subsample":
        best_result["subsample"],

    "colsample_bytree":
        best_result["colsample_bytree"],

    "reg_alpha":
        best_result["reg_alpha"],

    "reg_lambda":
        best_result["reg_lambda"]
}

best_params

In [ ]:
print(
    "기존 ClassWeight PR-AUC :",
    0.7360
)

print(
    "튜닝 LightGBM PR-AUC    :",
    round(
        best_result["PR_AUC_mean"],
        4
    )
)

print(
    "튜닝 Std                :",
    round(
        best_result["PR_AUC_std"],
        4
    )
)

In [ ]:
tuning_df[
    [
        "PR_AUC_mean",
        "PR_AUC_std",
        "n_estimators",
        "learning_rate",
        "num_leaves",
        "max_depth",
        "min_child_samples",
        "subsample",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda"
    ]
].head(10).round(4)

In [ ]:
best_params_cn7 = {
    "n_estimators": 200,
    "learning_rate": 0.10,
    "num_leaves": 15,
    "max_depth": 7,
    "min_child_samples": 10,
    "subsample": 1.0,
    "colsample_bytree": 0.70,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0
}

In [ ]:
SEEDS = [42, 100, 2026, 777, 1234]

oof_by_seed_tuned = {}

pr_auc_tuned_results = []

for seed in SEEDS:

    sgkf = StratifiedGroupKFold(
        n_splits=4,
        shuffle=True,
        random_state=seed
    )

    oof_proba = np.zeros(
        len(X_final)
    )

    for train_idx, val_idx in sgkf.split(
        X_final,
        y_final,
        groups=groups_final
    ):

        X_train = X_final.iloc[train_idx]
        X_val   = X_final.iloc[val_idx]

        y_train = y_final.iloc[train_idx]
        y_val   = y_final.iloc[val_idx]

        model = LGBMClassifier(
            **best_params_cn7,
            class_weight="balanced",
            random_state=42,
            verbose=-1
        )

        model.fit(
            X_train,
            y_train
        )

        oof_proba[val_idx] = (
            model.predict_proba(
                X_val
            )[:, 1]
        )

    oof_by_seed_tuned[seed] = oof_proba

    pr_auc = average_precision_score(
        y_final,
        oof_proba
    )

    pr_auc_tuned_results.append(
        pr_auc
    )

    print(
        f"Seed {seed} | PR-AUC: {pr_auc:.4f}"
    )

In [ ]:
print(
    "PR-AUC Mean:",
    np.mean(pr_auc_tuned_results)
)

print(
    "PR-AUC Std:",
    np.std(pr_auc_tuned_results)
)

### Threshold 재탐색

In [ ]:
threshold_results_tuned = []

thresholds = np.arange(
    0.005,
    0.501,
    0.005
)

for threshold in thresholds:

    seed_metrics = []

    for seed in SEEDS:

        proba = oof_by_seed_tuned[seed]

        pred = (
            proba >= threshold
        ).astype(int)

        precision = precision_score(
            y_final,
            pred,
            zero_division=0
        )

        recall = recall_score(
            y_final,
            pred,
            zero_division=0
        )

        f1 = f1_score(
            y_final,
            pred,
            zero_division=0
        )

        f2 = fbeta_score(
            y_final,
            pred,
            beta=2,
            zero_division=0
        )

        tn, fp, fn, tp = confusion_matrix(
            y_final,
            pred
        ).ravel()

        seed_metrics.append({
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "F2": f2,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn
        })

    temp = pd.DataFrame(seed_metrics)

    threshold_results_tuned.append({
        "Threshold": round(threshold, 3),
        "Precision_mean": temp["Precision"].mean(),
        "Recall_mean": temp["Recall"].mean(),
        "F1_mean": temp["F1"].mean(),
        "F2_mean": temp["F2"].mean(),
        "TP_mean": temp["TP"].mean(),
        "FP_mean": temp["FP"].mean(),
        "FN_mean": temp["FN"].mean()
    })

threshold_tuned_df = pd.DataFrame(
    threshold_results_tuned
)

In [ ]:
best_f1_tuned = threshold_tuned_df.loc[
    threshold_tuned_df["F1_mean"].idxmax()
]

best_f1_tuned

In [ ]:
best_f2_tuned = threshold_tuned_df.loc[
    threshold_tuned_df["F2_mean"].idxmax()
]

best_f2_tuned

In [ ]:
threshold_tuned_df.sort_values(
    "F1_mean",
    ascending=False
).head(10).round(4)

In [ ]:
threshold_tuned_df.sort_values(
    "F2_mean",
    ascending=False
).head(10).round(4)

| 항목 | 최종 설정 |
|---|---|
| Feature | **Original + Side_RH + Roll5 Mean** |
| Feature 수 | **47개** |
| Validation | **StratifiedGroupKFold** |
| Group | `TimeStamp` |
| Fold | 4 |
| 반복 Seed | 5개 |
| Model | **LightGBM** |
| 불균형 처리 | `class_weight="balanced"` |
| `n_estimators` | **200** |
| `learning_rate` | **0.10** |
| `num_leaves` | **15** |
| `max_depth` | **7** |
| `min_child_samples` | **10** |
| `subsample` | **1.0** |
| `colsample_bytree` | **0.70** |
| `reg_alpha` | **0.0** |
| `reg_lambda` | **0.0** |
| 반복 CV PR-AUC | **0.8157 ± 0.0139** |
| F1 최적 Threshold | **0.080** |
| 운영 Threshold 후보 | **0.060** |
| 운영 Precision | **0.7225** |
| 운영 Recall | **0.7882** |
| 운영 F1 | **0.7530** |
| 운영 F2 | **0.7735** |

# SHAP

In [ ]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lightgbm import LGBMClassifier

In [ ]:
final_model_cn7 = LGBMClassifier(
    **best_params_cn7,
    class_weight="balanced",
    random_state=42,
    verbose=-1
)

final_model_cn7.fit(
    X_final,
    y_final
)

In [ ]:
explainer_cn7 = shap.TreeExplainer(
    final_model_cn7
)

shap_values_cn7 = explainer_cn7(
    X_final
)

In [ ]:
print(shap_values_cn7.values.shape)
print(X_final.shape)

### 전체 feature 중요도

In [ ]:
shap.plots.bar(
    shap_values_cn7,
    max_display=20
)

### beeswarm

오른쪽 → 불량(1) 예측 방향으로 영향   
왼쪽 → 정상(0) 예측 방향으로 영향   
빨간색 → 해당 Feature 값이 높음   
파란색 → 해당 Feature 값이 낮음

In [ ]:
shap.plots.beeswarm(
    shap_values_cn7,
    max_display=20
)

### 중요도 dataframe

In [ ]:
shap_importance_cn7 = pd.DataFrame({
    "Feature": X_final.columns,
    "Mean_Abs_SHAP": np.abs(
        shap_values_cn7.values
    ).mean(axis=0)
})

shap_importance_cn7 = (
    shap_importance_cn7
    .sort_values(
        "Mean_Abs_SHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

shap_importance_cn7.head(20)

In [ ]:
shap_importance_cn7.head(20)

In [ ]:
shap_importance_cn7.to_csv(
    "CN7_SHAP_feature_importance.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
shap_importance_cn7["Feature_Type"] = (
    shap_importance_cn7["Feature"]
    .apply(
        lambda x:
        "Rolling Mean"
        if x.endswith("_roll5_mean")
        else (
            "Side"
            if x == "Side_RH"
            else "Original"
        )
    )
)

In [ ]:
feature_type_importance = (
    shap_importance_cn7
    .groupby("Feature_Type")[
        "Mean_Abs_SHAP"
    ]
    .sum()
    .sort_values(
        ascending=False
    )
)

feature_type_importance

In [ ]:
feature_type_ratio = (
    feature_type_importance
    / feature_type_importance.sum()
    * 100
)

feature_type_ratio.round(2)

In [ ]:
shap_sensor = shap_importance_cn7.copy()

shap_sensor["Base_Feature"] = (
    shap_sensor["Feature"]
    .str.replace(
        "_roll5_mean",
        "",
        regex=False
    )
)

sensor_importance_cn7 = (
    shap_sensor
    .groupby("Base_Feature")[
        "Mean_Abs_SHAP"
    ]
    .sum()
    .sort_values(
        ascending=False
    )
    .reset_index()
)

sensor_importance_cn7.head(15)

In [ ]:
top_sensor = (
    sensor_importance_cn7
    .head(15)
    .sort_values(
        "Mean_Abs_SHAP"
    )
)

plt.figure(figsize=(9, 7))

plt.barh(
    top_sensor["Base_Feature"],
    top_sensor["Mean_Abs_SHAP"]
)

plt.xlabel("Mean |SHAP value|")
plt.ylabel("Sensor")
plt.title(
    "CN7 Sensor-level SHAP Importance"
)

plt.tight_layout()
plt.show()

In [ ]:
top_feature_cn7 = (
    shap_importance_cn7
    .iloc[0]["Feature"]
)

top_feature_cn7

In [ ]:
defect_mask = (
    y_final.values == 1
)

shap_defect_cn7 = (
    shap_values_cn7[
        defect_mask
    ]
)

shap.plots.beeswarm(
    shap_defect_cn7,
    max_display=20
)

# SAHP 분석 결과

### 1. feature 유형별 shap 중요도

| Feature Type | SHAP 중요도 비율 |
|---|---:|
| **Rolling Mean** | **59.85%** |
| Original | 29.63% |
| Side | 10.52% |


### 2. shap top 20 feature
| Rank | Feature | Mean \|SHAP\| | 구분 |
|---:|---|---:|---|
| 1 | **Side_RH** | **0.6191** | Side |
| 2 | **Mold_Temperature_3_roll5_mean** | **0.5418** | Rolling |
| 3 | **Plasticizing_Position** | **0.4676** | Original |
| 4 | **Max_Screw_RPM_roll5_mean** | **0.4523** | Rolling |
| 5 | **Mold_Temperature_4_roll5_mean** | **0.4298** | Rolling |
| 6 | Hopper_Temperature_roll5_mean | 0.3578 | Rolling |
| 7 | Clamp_Close_Time | 0.3551 | Original |
| 8 | Barrel_Temperature_2_roll5_mean | 0.3546 | Rolling |
| 9 | Clamp_Close_Time_roll5_mean | 0.3169 | Rolling |
| 10 | Barrel_Temperature_5_roll5_mean | 0.1687 | Rolling |
| 11 | Max_Injection_Speed | 0.1527 | Original |
| 12 | Injection_Time | 0.1409 | Original |
| 13 | Plasticizing_Time_roll5_mean | 0.1380 | Rolling |
| 14 | Filling_Time_roll5_mean | 0.1262 | Rolling |
| 15 | Barrel_Temperature_2 | 0.1203 | Original |
| 16 | Plasticizing_Time | 0.0973 | Original |
| 17 | Barrel_Temperature_6_roll5_mean | 0.0967 | Rolling |
| 18 | Cycle_Time_roll5_mean | 0.0900 | Rolling |
| 19 | Barrel_Temperature_1_roll5_mean | 0.0809 | Rolling |
| 20 | Filling_Time | 0.0803 | Original |




In [ ]:
shap.plots.beeswarm(
    shap_values_cn7,
    max_display=20
)

In [ ]:
top_process_features = [
    "Mold_Temperature_3_roll5_mean",
    "Plasticizing_Position",
    "Max_Screw_RPM_roll5_mean",
    "Mold_Temperature_4_roll5_mean",
    "Hopper_Temperature_roll5_mean",
    "Clamp_Close_Time"
]

for feature in top_process_features:
    print("\n", feature)

    shap.plots.scatter(
        shap_values_cn7[:, feature]
    )

| Feature | 관찰된 SHAP 패턴 | 해석 |
|---|---|---|
| **Side_RH** | RH에 해당하는 높은 값이 주로 +SHAP | **RH일 때 불량 방향으로 판단하는 경향** |
| **Mold_Temperature_3_roll5_mean** | 높은 구간에서 SHAP가 급격히 + | 최근 금형온도3 평균이 높은 구간에서 **불량 위험 증가** |
| **Plasticizing_Position** | 낮은 값은 -SHAP, 높은 값은 +SHAP | 가소화 위치가 높은 영역에서 **불량 방향 영향** |
| **Max_Screw_RPM_roll5_mean** | 단순 직선 관계가 아닌 U/비선형 패턴 | 평균 Screw RPM이 특정 중간 구간을 벗어나면 위험 증가 가능 |
| **Mold_Temperature_4_roll5_mean** | 낮은 영역은 -, 높은 영역은 강한 + | 최근 금형온도4가 높은 상태에서 **불량 위험 크게 증가** |
| **Hopper_Temperature_roll5_mean** | 강한 비선형 관계 | 특정 낮은 온도 영역에서 위험이 크게 변화 |
| **Clamp_Close_Time** | 낮은 값은 -, 높은 값은 + | Clamp Close Time이 상대적으로 길면 **불량 방향 영향** |

# CN7 Side_RH 제거 Ablation 분석

In [ ]:
# ============================================================
# CN7 Side_RH 제거 Ablation 분석
# - 기존 최종 모델: V5 47 Features + ClassWeight + Tuned LightGBM
# - 목적: Side_RH가 모델 성능에 실제로 얼마나 기여하는지 검증
# ============================================================

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)


# ============================================================
# 1. Side_RH 제거
# ============================================================

X_cn7_no_side = X_final.drop(
    columns=["Side_RH"]
).copy()

print("기존 Feature 수 :", X_final.shape[1])
print("Side 제거 후    :", X_cn7_no_side.shape[1])


# ============================================================
# 2. 기존 최적 LightGBM 파라미터
# ============================================================

best_params_cn7 = {
    "n_estimators": 200,
    "learning_rate": 0.10,
    "num_leaves": 15,
    "max_depth": 7,
    "min_child_samples": 10,
    "subsample": 1.0,
    "colsample_bytree": 0.70,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0
}


# ============================================================
# 3. 반복 검증 설정
# ============================================================

SEEDS = [
    42,
    100,
    2026,
    777,
    1234
]

N_SPLITS = 4

oof_no_side = {}
pr_auc_no_side = []


# ============================================================
# 4. Side_RH 제거 모델
#    5 Seeds × 4 Fold StratifiedGroupKFold
# ============================================================

for seed in SEEDS:

    sgkf = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    oof_proba = np.zeros(
        len(X_cn7_no_side)
    )

    for fold, (train_idx, val_idx) in enumerate(
        sgkf.split(
            X_cn7_no_side,
            y_final,
            groups=groups_final
        ),
        start=1
    ):

        X_train = X_cn7_no_side.iloc[
            train_idx
        ].copy()

        X_val = X_cn7_no_side.iloc[
            val_idx
        ].copy()

        y_train = y_final.iloc[
            train_idx
        ].copy()

        y_val = y_final.iloc[
            val_idx
        ].copy()


        model = LGBMClassifier(
            **best_params_cn7,
            class_weight="balanced",
            random_state=42,
            verbose=-1
        )


        model.fit(
            X_train,
            y_train
        )


        val_proba = (
            model
            .predict_proba(
                X_val
            )[:, 1]
        )


        oof_proba[
            val_idx
        ] = val_proba


    # Seed별 OOF 저장
    oof_no_side[
        seed
    ] = oof_proba


    # Seed별 PR-AUC
    seed_pr_auc = (
        average_precision_score(
            y_final,
            oof_proba
        )
    )

    pr_auc_no_side.append(
        seed_pr_auc
    )


    print(
        f"Seed {seed} | "
        f"PR-AUC: {seed_pr_auc:.4f}"
    )


# ============================================================
# 5. Side 제거 PR-AUC 요약
# ============================================================

no_side_mean = np.mean(
    pr_auc_no_side
)

no_side_std = np.std(
    pr_auc_no_side
)

no_side_min = np.min(
    pr_auc_no_side
)

no_side_max = np.max(
    pr_auc_no_side
)


print("\n" + "=" * 60)

print("Side 제거 PR-AUC")

print("=" * 60)

print(
    f"Mean : {no_side_mean:.4f}"
)

print(
    f"Std  : {no_side_std:.4f}"
)

print(
    f"Min  : {no_side_min:.4f}"
)

print(
    f"Max  : {no_side_max:.4f}"
)


# ============================================================
# 6. 기존 Side 포함 모델과 PR-AUC 비교
# ============================================================

full_mean = (
    0.8157169084660907
)

full_std = (
    0.013893581153823713
)


comparison_side = pd.DataFrame({
    "Model": [
        "With Side_RH",
        "Without Side_RH"
    ],

    "Features": [
        47,
        46
    ],

    "PR_AUC_mean": [
        full_mean,
        no_side_mean
    ],

    "PR_AUC_std": [
        full_std,
        no_side_std
    ]
})


comparison_side[
    "Delta_vs_Full"
] = (
    comparison_side[
        "PR_AUC_mean"
    ]
    - full_mean
)


print("\n[Side Ablation - PR-AUC 비교]")

display(
    comparison_side.round(4)
)


# ============================================================
# 7. Side 제거 모델 Threshold 탐색
# ============================================================

thresholds = np.arange(
    0.005,
    0.501,
    0.005
)

threshold_results_no_side = []


for threshold in thresholds:

    seed_metrics = []


    for seed in SEEDS:

        proba = (
            oof_no_side[
                seed
            ]
        )


        pred = (
            proba >= threshold
        ).astype(int)


        precision = (
            precision_score(
                y_final,
                pred,
                zero_division=0
            )
        )


        recall = (
            recall_score(
                y_final,
                pred,
                zero_division=0
            )
        )


        f1 = (
            f1_score(
                y_final,
                pred,
                zero_division=0
            )
        )


        f2 = (
            fbeta_score(
                y_final,
                pred,
                beta=2,
                zero_division=0
            )
        )


        tn, fp, fn, tp = (
            confusion_matrix(
                y_final,
                pred
            ).ravel()
        )


        seed_metrics.append({
            "Precision":
                precision,

            "Recall":
                recall,

            "F1":
                f1,

            "F2":
                f2,

            "TP":
                tp,

            "FP":
                fp,

            "FN":
                fn,

            "TN":
                tn
        })


    temp = pd.DataFrame(
        seed_metrics
    )


    threshold_results_no_side.append({

        "Threshold":
            round(
                threshold,
                3
            ),

        "Precision_mean":
            temp[
                "Precision"
            ].mean(),

        "Recall_mean":
            temp[
                "Recall"
            ].mean(),

        "F1_mean":
            temp[
                "F1"
            ].mean(),

        "F2_mean":
            temp[
                "F2"
            ].mean(),

        "TP_mean":
            temp[
                "TP"
            ].mean(),

        "FP_mean":
            temp[
                "FP"
            ].mean(),

        "FN_mean":
            temp[
                "FN"
            ].mean()
    })


threshold_no_side_df = pd.DataFrame(
    threshold_results_no_side
)


# ============================================================
# 8. Side 제거 모델 F1 최고 Threshold
# ============================================================

best_f1_no_side = (
    threshold_no_side_df
    .loc[
        threshold_no_side_df[
            "F1_mean"
        ].idxmax()
    ]
)


print(
    "\n[F1 기준 최적 Threshold]"
)

display(
    best_f1_no_side
    .to_frame()
    .T
    .round(4)
)


# ============================================================
# 9. Side 제거 모델 F2 최고 Threshold
# ============================================================

best_f2_no_side = (
    threshold_no_side_df
    .loc[
        threshold_no_side_df[
            "F2_mean"
        ].idxmax()
    ]
)


print(
    "\n[F2 기준 최적 Threshold]"
)

display(
    best_f2_no_side
    .to_frame()
    .T
    .round(4)
)


# ============================================================
# 10. F1 기준 상위 Threshold 10개
# ============================================================

print(
    "\n[F1 기준 Threshold TOP 10]"
)

display(
    threshold_no_side_df
    .sort_values(
        "F1_mean",
        ascending=False
    )
    .head(10)
    .round(4)
)


# ============================================================
# 11. 최종 Side Ablation 비교
# ============================================================

# Side 포함 모델의 기존 F1 최고 성능
side_f1_threshold = 0.080
side_precision = 0.7508
side_recall = 0.7647
side_f1 = 0.7558
side_f2 = 0.7606


side_ablation_final = pd.DataFrame({

    "Model": [
        "With Side_RH",
        "Without Side_RH"
    ],

    "Features": [
        47,
        46
    ],

    "PR_AUC": [
        full_mean,
        no_side_mean
    ],

    "PR_AUC_Std": [
        full_std,
        no_side_std
    ],

    "Threshold": [
        side_f1_threshold,
        best_f1_no_side[
            "Threshold"
        ]
    ],

    "Precision": [
        side_precision,
        best_f1_no_side[
            "Precision_mean"
        ]
    ],

    "Recall": [
        side_recall,
        best_f1_no_side[
            "Recall_mean"
        ]
    ],

    "F1": [
        side_f1,
        best_f1_no_side[
            "F1_mean"
        ]
    ],

    "F2": [
        side_f2,
        best_f1_no_side[
            "F2_mean"
        ]
    ]
})


side_ablation_final[
    "PR_AUC_Delta"
] = (
    side_ablation_final[
        "PR_AUC"
    ]
    - full_mean
)


print(
    "\n"
    + "=" * 60
)

print(
    "CN7 FINAL SIDE ABLATION RESULT"
)

print(
    "=" * 60
)


display(
    side_ablation_final
    .round(4)
)

# CN7 Side_RH 단독 모델 검증

In [ ]:
# ============================================================
# CN7 Side_RH Only Sanity Check
# 목적:
# Side_RH 하나만으로 모델 성능이 어느 정도 나오는지 확인
# ============================================================

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)


# ============================================================
# 1. Side_RH 하나만 사용
# ============================================================

X_side_only = X_final[
    ["Side_RH"]
].copy()

print("Side-only Feature 수 :", X_side_only.shape[1])
print("Feature :", X_side_only.columns.tolist())


# ============================================================
# 2. 기존 최적 LightGBM 파라미터 그대로 사용
# ============================================================

best_params_cn7 = {
    "n_estimators": 200,
    "learning_rate": 0.10,
    "num_leaves": 15,
    "max_depth": 7,
    "min_child_samples": 10,
    "subsample": 1.0,
    "colsample_bytree": 0.70,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0
}


# ============================================================
# 3. 반복 검증 설정
# ============================================================

SEEDS = [
    42,
    100,
    2026,
    777,
    1234
]

N_SPLITS = 4

oof_side_only = {}
pr_auc_side_only = []


# ============================================================
# 4. 5 Seeds × 4 Fold Group CV
# ============================================================

for seed in SEEDS:

    sgkf = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    oof_proba = np.zeros(
        len(X_side_only)
    )

    for train_idx, val_idx in sgkf.split(
        X_side_only,
        y_final,
        groups=groups_final
    ):

        X_train = X_side_only.iloc[
            train_idx
        ].copy()

        X_val = X_side_only.iloc[
            val_idx
        ].copy()

        y_train = y_final.iloc[
            train_idx
        ].copy()

        y_val = y_final.iloc[
            val_idx
        ].copy()


        model = LGBMClassifier(
            **best_params_cn7,
            class_weight="balanced",
            random_state=42,
            verbose=-1
        )


        model.fit(
            X_train,
            y_train
        )


        val_proba = (
            model
            .predict_proba(
                X_val
            )[:, 1]
        )


        oof_proba[
            val_idx
        ] = val_proba


    oof_side_only[
        seed
    ] = oof_proba


    seed_pr_auc = (
        average_precision_score(
            y_final,
            oof_proba
        )
    )

    pr_auc_side_only.append(
        seed_pr_auc
    )


    print(
        f"Seed {seed} | "
        f"PR-AUC: {seed_pr_auc:.4f}"
    )


# ============================================================
# 5. Side-only PR-AUC 요약
# ============================================================

side_only_mean = np.mean(
    pr_auc_side_only
)

side_only_std = np.std(
    pr_auc_side_only
)


print("\n" + "=" * 60)

print("Side_RH ONLY PR-AUC")

print("=" * 60)

print(
    f"Mean : {side_only_mean:.4f}"
)

print(
    f"Std  : {side_only_std:.4f}"
)

print(
    f"Min  : {np.min(pr_auc_side_only):.4f}"
)

print(
    f"Max  : {np.max(pr_auc_side_only):.4f}"
)


# ============================================================
# 6. Side-only Threshold 탐색
# ============================================================

thresholds = np.arange(
    0.005,
    0.501,
    0.005
)

threshold_side_only_results = []


for threshold in thresholds:

    seed_metrics = []


    for seed in SEEDS:

        proba = (
            oof_side_only[
                seed
            ]
        )


        pred = (
            proba >= threshold
        ).astype(int)


        precision = (
            precision_score(
                y_final,
                pred,
                zero_division=0
            )
        )


        recall = (
            recall_score(
                y_final,
                pred,
                zero_division=0
            )
        )


        f1 = (
            f1_score(
                y_final,
                pred,
                zero_division=0
            )
        )


        f2 = (
            fbeta_score(
                y_final,
                pred,
                beta=2,
                zero_division=0
            )
        )


        tn, fp, fn, tp = (
            confusion_matrix(
                y_final,
                pred
            ).ravel()
        )


        seed_metrics.append({
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "F2": f2,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn
        })


    temp = pd.DataFrame(
        seed_metrics
    )


    threshold_side_only_results.append({

        "Threshold":
            round(
                threshold,
                3
            ),

        "Precision_mean":
            temp[
                "Precision"
            ].mean(),

        "Recall_mean":
            temp[
                "Recall"
            ].mean(),

        "F1_mean":
            temp[
                "F1"
            ].mean(),

        "F2_mean":
            temp[
                "F2"
            ].mean(),

        "TP_mean":
            temp[
                "TP"
            ].mean(),

        "FP_mean":
            temp[
                "FP"
            ].mean(),

        "FN_mean":
            temp[
                "FN"
            ].mean()
    })


threshold_side_only_df = pd.DataFrame(
    threshold_side_only_results
)


# ============================================================
# 7. Side-only F1 최고 Threshold
# ============================================================

best_f1_side_only = (
    threshold_side_only_df
    .loc[
        threshold_side_only_df[
            "F1_mean"
        ].idxmax()
    ]
)


print(
    "\n[F1 기준 Side-only 최적 Threshold]"
)

print(
    best_f1_side_only
    .round(4)
    .to_string()
)


# ============================================================
# 8. 최종 비교
# ============================================================

comparison_final = pd.DataFrame({

    "Model": [
        "Full Model",
        "Without Side_RH",
        "Side_RH Only"
    ],

    "Features": [
        47,
        46,
        1
    ],

    "PR_AUC": [
        0.8157,
        0.6509,
        side_only_mean
    ],

    "Precision": [
        0.7508,
        0.6093,
        best_f1_side_only[
            "Precision_mean"
        ]
    ],

    "Recall": [
        0.7647,
        0.5647,
        best_f1_side_only[
            "Recall_mean"
        ]
    ],

    "F1": [
        0.7558,
        0.5843,
        best_f1_side_only[
            "F1_mean"
        ]
    ],

    "F2": [
        0.7606,
        0.5720,
        best_f1_side_only[
            "F2_mean"
        ]
    ]
})


print(
    "\n" + "=" * 90
)

print(
    "CN7 SIDE SANITY CHECK FINAL"
)

print(
    "=" * 90
)


print(
    comparison_final
    .round(4)
    .to_string(index=False)
)

# 최종 모델 저장

In [ ]:
import os
import joblib

os.makedirs("models", exist_ok=True)

cn7_artifact = {
    "model": final_model_cn7,
    "features": X_final.columns.tolist(),
    "threshold_f1": 0.080,
    "threshold_operation": 0.060,
    "best_params": best_params_cn7,
    "pr_auc_mean": 0.8157169084660907,
    "pr_auc_std": 0.013893581153823713
}

joblib.dump(
    cn7_artifact,
    "models/cn7_lgbm_final.joblib"
)

print("저장 완료: models/cn7_lgbm_final.joblib")

In [ ]:
cn7_loaded = joblib.load(
    "models/cn7_lgbm_final.joblib"
)

model = cn7_loaded["model"]
features = cn7_loaded["features"]
threshold = cn7_loaded["threshold_operation"]

print("Feature 수:", len(features))
print("Threshold:", threshold)

# CN7 Cost-Ratio Sensitivity Analysis

In [ ]:
# ============================================================
# CN7 Cost-Ratio Sensitivity Analysis
#
# 목적
# - 실제 비용을 임의로 가정하지 않고
# - FN 비용 / FP 비용 비율에 따라
#   최적 Threshold가 어떻게 달라지는지 분석
#
# Relative Cost = FP + Cost_Ratio * FN
#
# Cost_Ratio = C_FN / C_FP
# 예)
# 1   -> FP와 FN을 동일하게 평가
# 5   -> FN을 FP보다 5배 중요하게 평가
# 20  -> FN을 FP보다 20배 중요하게 평가
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. 설정
# ------------------------------------------------------------

SEEDS = [
    42,
    100,
    2026,
    777,
    1234
]

# FN 비용 / FP 비용
COST_RATIOS = [
    1,
    2,
    5,
    10,
    20,
    50,
    100
]

thresholds = np.arange(
    0.005,
    0.501,
    0.005
)

N = len(y_final)


# ------------------------------------------------------------
# 2. Cost Ratio × Threshold 전체 탐색
# ------------------------------------------------------------

cost_results = []

for cost_ratio in COST_RATIOS:

    for threshold in thresholds:

        seed_results = []

        for seed in SEEDS:

            proba = oof_by_seed_tuned[seed]

            pred = (
                proba >= threshold
            ).astype(int)

            precision = precision_score(
                y_final,
                pred,
                zero_division=0
            )

            recall = recall_score(
                y_final,
                pred,
                zero_division=0
            )

            f1 = f1_score(
                y_final,
                pred,
                zero_division=0
            )

            f2 = fbeta_score(
                y_final,
                pred,
                beta=2,
                zero_division=0
            )

            tn, fp, fn, tp = confusion_matrix(
                y_final,
                pred
            ).ravel()

            # ----------------------------------
            # 상대 비용
            # FP 비용을 1로 기준
            # ----------------------------------

            relative_cost = (
                fp
                + cost_ratio * fn
            )

            # ----------------------------------
            # 검사 후보 비율
            # 모델이 불량이라고 Flag한 비율
            # ----------------------------------

            flag_rate = (
                (tp + fp) / N
            )

            seed_results.append({
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
                "F2": f2,

                "TP": tp,
                "FP": fp,
                "FN": fn,
                "TN": tn,

                "Relative_Cost":
                    relative_cost,

                "Flag_Rate":
                    flag_rate
            })


        temp = pd.DataFrame(
            seed_results
        )

        cost_results.append({

            "Cost_Ratio_FN_to_FP":
                cost_ratio,

            "Threshold":
                round(threshold, 3),

            "Precision_mean":
                temp["Precision"].mean(),

            "Recall_mean":
                temp["Recall"].mean(),

            "F1_mean":
                temp["F1"].mean(),

            "F2_mean":
                temp["F2"].mean(),

            "TP_mean":
                temp["TP"].mean(),

            "FP_mean":
                temp["FP"].mean(),

            "FN_mean":
                temp["FN"].mean(),

            "Relative_Cost_mean":
                temp["Relative_Cost"].mean(),

            "Flag_Rate_mean":
                temp["Flag_Rate"].mean()
        })


cost_ratio_df = pd.DataFrame(
    cost_results
)


# ------------------------------------------------------------
# 3. 비용비별 최소 Cost Threshold
# ------------------------------------------------------------

best_cost_by_ratio = (
    cost_ratio_df
    .loc[
        cost_ratio_df
        .groupby(
            "Cost_Ratio_FN_to_FP"
        )[
            "Relative_Cost_mean"
        ]
        .idxmin()
    ]
    .sort_values(
        "Cost_Ratio_FN_to_FP"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. 검사 후보 비율을 %로 변환
# ------------------------------------------------------------

best_cost_by_ratio[
    "Flag_Rate_pct"
] = (
    best_cost_by_ratio[
        "Flag_Rate_mean"
    ]
    * 100
)


# ------------------------------------------------------------
# 5. 최종 결과
# ------------------------------------------------------------

final_cost_table = (
    best_cost_by_ratio[
        [
            "Cost_Ratio_FN_to_FP",
            "Threshold",
            "Precision_mean",
            "Recall_mean",
            "F1_mean",
            "F2_mean",
            "TP_mean",
            "FP_mean",
            "FN_mean",
            "Flag_Rate_pct",
            "Relative_Cost_mean"
        ]
    ]
)


print(
    "\n"
    + "=" * 110
)

print(
    "CN7 COST-RATIO SENSITIVITY ANALYSIS"
)

print(
    "=" * 110
)


print(
    final_cost_table
    .round(4)
    .to_string(index=False)
)

| FN:FP 비용비 | 최적 Threshold | Precision | Recall | F1 | 평균 TP | 평균 FP | 평균 FN | 검사 후보 비율 |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| **1:1** | **0.470** | **0.8206** | 0.6824 | 0.7420 | 11.6 | **2.6** | 5.4 | **1.17%** |
| **2:1** | **0.060** | 0.7225 | 0.7882 | **0.7530** | 13.4 | 5.2 | 3.6 | **1.54%** |
| **5:1** | **0.020** | 0.6788 | **0.8118** | 0.7380 | 13.8 | 6.6 | **3.2** | **1.68%** |
| **10:1** | **0.005*** | 0.5895 | **0.8353** | 0.6891 | **14.2** | 10.0 | **2.8** | **2.00%** |
| 20:1 | 0.005* | 0.5895 | 0.8353 | 0.6891 | 14.2 | 10.0 | 2.8 | 2.00% |
| 50:1 | 0.005* | 0.5895 | 0.8353 | 0.6891 | 14.2 | 10.0 | 2.8 | 2.00% |
| 100:1 | 0.005* | 0.5895 | 0.8353 | 0.6891 | 14.2 | 10.0 | 2.8 | 2.00% |

미탐(FN)을 더 중요하게 평가할수록 더 공격적으로 불량 후보를 Flag하는 구조   
미탐(FN)을 더 중요하게 평가할수록 더 공격적으로 불량 후보를 Flag하는 구조   
미탐을 오탐보다 더 중요하게 보는 제조 품질관리 상황에서, 2:1 비용비 가정 시 상대비용이 최소화되는 Threshold로도 확인

In [ ]:
# ============================================
# 높은 FN 비용비용 Low Threshold 추가 탐색
# ============================================

thresholds_low = np.concatenate([
    np.arange(0.0001, 0.005, 0.0001),
    np.arange(0.005, 0.101, 0.005)
])

COST_RATIOS_HIGH = [
    10,
    20,
    50,
    100
]

low_cost_results = []

for cost_ratio in COST_RATIOS_HIGH:

    for threshold in thresholds_low:

        seed_results = []

        for seed in SEEDS:

            proba = oof_by_seed_tuned[seed]

            pred = (
                proba >= threshold
            ).astype(int)

            precision = precision_score(
                y_final,
                pred,
                zero_division=0
            )

            recall = recall_score(
                y_final,
                pred,
                zero_division=0
            )

            f1 = f1_score(
                y_final,
                pred,
                zero_division=0
            )

            tn, fp, fn, tp = confusion_matrix(
                y_final,
                pred
            ).ravel()

            cost = (
                fp
                + cost_ratio * fn
            )

            flag_rate = (
                (tp + fp)
                / len(y_final)
            )

            seed_results.append({
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
                "TP": tp,
                "FP": fp,
                "FN": fn,
                "Cost": cost,
                "Flag_Rate": flag_rate
            })

        temp = pd.DataFrame(
            seed_results
        )

        low_cost_results.append({
            "Cost_Ratio": cost_ratio,
            "Threshold": threshold,
            "Precision": temp["Precision"].mean(),
            "Recall": temp["Recall"].mean(),
            "F1": temp["F1"].mean(),
            "TP": temp["TP"].mean(),
            "FP": temp["FP"].mean(),
            "FN": temp["FN"].mean(),
            "Flag_Rate_pct":
                temp["Flag_Rate"].mean() * 100,
            "Relative_Cost":
                temp["Cost"].mean()
        })


low_cost_df = pd.DataFrame(
    low_cost_results
)

best_low_cost = (
    low_cost_df
    .loc[
        low_cost_df
        .groupby("Cost_Ratio")[
            "Relative_Cost"
        ]
        .idxmin()
    ]
    .sort_values("Cost_Ratio")
)

print(
    best_low_cost
    .round(4)
    .to_string(index=False)
)

| FN:FP 비용비 | 최적 Threshold | Precision | Recall | F1 | 평균 FN | 평균 FP | 검사 후보 비율 |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 1:1 | 0.4700 | **82.06%** | 68.24% | 74.20% | 5.4 | **2.6** | **1.17%** |
| 2:1 | 0.0600 | 72.25% | 78.82% | **75.30%** | 3.6 | 5.2 | 1.54% |
| 5:1 | 0.0200 | 67.88% | 81.18% | 73.80% | 3.2 | 6.6 | 1.68% |
| 10:1 | 0.0005 | 47.96% | 88.24% | 62.05% | 2.0 | 16.4 | 2.59% |
| 20:1 | 0.0002 | 40.88% | 91.76% | 56.51% | 1.4 | 22.8 | 3.17% |
| 50:1 | 0.0001* | 34.27% | **92.94%** | 50.06% | **1.2** | 30.4 | 3.82% |
| 100:1 | 0.0001* | 34.27% | **92.94%** | 50.06% | **1.2** | 30.4 | 3.82% |

| 운영 모드 | 비용비 예시 | Threshold | Precision | Recall | 검사 후보 |
|---|---:|---:|---:|---:|---:|
| **정밀 검사 최소화형** | 1:1 | 0.470 | **82.1%** | 68.2% | **1.17%** |
| **균형 운영형** | 2:1 | **0.060** | 72.3% | **78.8%** | **1.54%** |
| **불량 미탐 최소화형** | 20:1 | 0.0002 | 40.9% | **91.8%** | 3.17% |

# CN7 오류분석 + 불량 위험도 + 검사 우선순위 생성

In [ ]:
# ============================================================
# CN7 품질 위험도 기반 오류분석 및 검사 우선순위 시스템
#
# 목적
# 1. 최종 품질검사 이전 불량 위험도 산출
# 2. 모델이 잘 맞는 조건 / 실패하는 조건 분석
# 3. Risk Score 기반 검사 우선순위 생성
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ============================================================
# 0. 기본 설정
# ============================================================

SEEDS = [42, 100, 2026, 777, 1234]

# 현재 CN7 균형 운영 Threshold
OPERATION_THRESHOLD = 0.060

# 기존 Cost-ratio 분석에서 얻은 기준
HIGH_PRECISION_THRESHOLD = 0.470
HIGH_RECALL_THRESHOLD = 0.020


# ============================================================
# 1. 5개 Seed OOF 확률 통합
# ============================================================

oof_matrix = np.column_stack([
    oof_by_seed_tuned[seed]
    for seed in SEEDS
])

# 제품별 평균 불량 확률
risk_proba = oof_matrix.mean(axis=1)

# Seed 간 예측 변동성
risk_std = oof_matrix.std(axis=1)

# Threshold 0.06 기준 Seed별 불량 예측 횟수
positive_votes = (
    oof_matrix >= OPERATION_THRESHOLD
).sum(axis=1)


# ============================================================
# 2. 분석 DataFrame 생성
# ============================================================

cn7_analysis = X_final.copy()

cn7_analysis["TimeStamp"] = groups_final.values
cn7_analysis["Actual"] = y_final.values

cn7_analysis["Risk_Probability"] = risk_proba

cn7_analysis["Risk_Score"] = (
    risk_proba * 100
)

cn7_analysis["Risk_STD"] = risk_std

cn7_analysis["Positive_Votes"] = positive_votes


# ============================================================
# 3. 최종 예측
# ============================================================

cn7_analysis["Predicted"] = (
    cn7_analysis["Risk_Probability"]
    >= OPERATION_THRESHOLD
).astype(int)


# ============================================================
# 4. TP / FP / FN / TN 구분
# ============================================================

conditions = [
    (
        (cn7_analysis["Actual"] == 1)
        & (cn7_analysis["Predicted"] == 1)
    ),

    (
        (cn7_analysis["Actual"] == 0)
        & (cn7_analysis["Predicted"] == 1)
    ),

    (
        (cn7_analysis["Actual"] == 1)
        & (cn7_analysis["Predicted"] == 0)
    ),

    (
        (cn7_analysis["Actual"] == 0)
        & (cn7_analysis["Predicted"] == 0)
    )
]

labels = [
    "TP",
    "FP",
    "FN",
    "TN"
]

cn7_analysis["Result_Type"] = np.select(
    conditions,
    labels,
    default="Unknown"
)


# ============================================================
# 5. 위험 등급
# ============================================================

def risk_level(prob):

    if prob >= HIGH_PRECISION_THRESHOLD:
        return "CRITICAL"

    elif prob >= OPERATION_THRESHOLD:
        return "HIGH"

    elif prob >= HIGH_RECALL_THRESHOLD:
        return "WATCH"

    else:
        return "LOW"


cn7_analysis["Risk_Level"] = (
    cn7_analysis[
        "Risk_Probability"
    ].apply(risk_level)
)


# ============================================================
# 6. 예측 안정성
#
# 5개 Seed가 같은 결론이면 Stable
# Seed마다 판단이 갈리면 Unstable
# ============================================================

def prediction_stability(votes):

    if votes == 0 or votes == len(SEEDS):
        return "Stable"

    else:
        return "Unstable"


cn7_analysis["Prediction_Stability"] = (
    cn7_analysis[
        "Positive_Votes"
    ].apply(prediction_stability)
)


# ============================================================
# 7. 검사 우선순위 생성
# ============================================================

cn7_analysis[
    "Inspection_Rank"
] = (
    cn7_analysis[
        "Risk_Probability"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)


def inspection_action(level):

    if level == "CRITICAL":
        return "즉시 검사"

    elif level == "HIGH":
        return "우선 검사"

    elif level == "WATCH":
        return "주의 / 추가 확인"

    else:
        return "일반 검사"


cn7_analysis[
    "Inspection_Action"
] = (
    cn7_analysis[
        "Risk_Level"
    ].apply(
        inspection_action
    )
)


# ============================================================
# 8. 전체 성능 확인
# ============================================================

precision = precision_score(
    cn7_analysis["Actual"],
    cn7_analysis["Predicted"],
    zero_division=0
)

recall = recall_score(
    cn7_analysis["Actual"],
    cn7_analysis["Predicted"],
    zero_division=0
)

f1 = f1_score(
    cn7_analysis["Actual"],
    cn7_analysis["Predicted"],
    zero_division=0
)

tn, fp, fn, tp = confusion_matrix(
    cn7_analysis["Actual"],
    cn7_analysis["Predicted"]
).ravel()


print("\n" + "=" * 80)
print("CN7 OOF QUALITY RISK MODEL")
print("=" * 80)

print(
    f"Threshold : {OPERATION_THRESHOLD}"
)

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)

print(
    f"F1        : {f1:.4f}"
)

print()

print(
    f"TP : {tp}"
)

print(
    f"FP : {fp}"
)

print(
    f"FN : {fn}"
)

print(
    f"TN : {tn}"
)


# ============================================================
# 9. Side별 성능 분석
# ============================================================

side_results = []

for side_value, temp in (
    cn7_analysis
    .groupby("Side_RH")
):

    precision_side = precision_score(
        temp["Actual"],
        temp["Predicted"],
        zero_division=0
    )

    recall_side = recall_score(
        temp["Actual"],
        temp["Predicted"],
        zero_division=0
    )

    f1_side = f1_score(
        temp["Actual"],
        temp["Predicted"],
        zero_division=0
    )

    side_results.append({

        "Side":
            "RH"
            if side_value == 1
            else "LH",

        "Count":
            len(temp),

        "Defects":
            temp["Actual"].sum(),

        "Precision":
            precision_side,

        "Recall":
            recall_side,

        "F1":
            f1_side
    })


side_performance = pd.DataFrame(
    side_results
)


print("\n" + "=" * 80)
print("SIDE별 모델 성능")
print("=" * 80)

print(
    side_performance
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 10. 위험등급별 실제 불량률 분석
# ============================================================

risk_level_summary = (
    cn7_analysis
    .groupby(
        "Risk_Level",
        observed=True
    )
    .agg(

        Count=(
            "Actual",
            "size"
        ),

        Actual_Defects=(
            "Actual",
            "sum"
        ),

        Actual_Defect_Rate=(
            "Actual",
            "mean"
        ),

        Avg_Risk_Score=(
            "Risk_Score",
            "mean"
        )
    )
    .reset_index()
)


risk_order = [
    "CRITICAL",
    "HIGH",
    "WATCH",
    "LOW"
]

risk_level_summary[
    "Risk_Level"
] = pd.Categorical(
    risk_level_summary[
        "Risk_Level"
    ],
    categories=risk_order,
    ordered=True
)

risk_level_summary = (
    risk_level_summary
    .sort_values(
        "Risk_Level"
    )
)


print("\n" + "=" * 80)
print("위험등급별 실제 불량 분포")
print("=" * 80)

print(
    risk_level_summary
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 11. 모델 안정 / 불안정 조건 분석
# ============================================================

stability_summary = (
    cn7_analysis
    .groupby(
        "Prediction_Stability"
    )
    .agg(

        Count=(
            "Actual",
            "size"
        ),

        Errors=(
            "Result_Type",
            lambda x:
                x.isin(
                    ["FP", "FN"]
                ).sum()
        ),

        Mean_Risk_STD=(
            "Risk_STD",
            "mean"
        )
    )
    .reset_index()
)


stability_summary[
    "Error_Rate"
] = (
    stability_summary[
        "Errors"
    ]
    / stability_summary[
        "Count"
    ]
)


print("\n" + "=" * 80)
print("예측 안정성별 오류율")
print("=" * 80)

print(
    stability_summary
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 12. TP / FP / FN / TN 분포
# ============================================================

error_summary = (
    cn7_analysis[
        "Result_Type"
    ]
    .value_counts()
    .reindex(
        [
            "TP",
            "FP",
            "FN",
            "TN"
        ],
        fill_value=0
    )
    .reset_index()
)

error_summary.columns = [
    "Result_Type",
    "Count"
]


print("\n" + "=" * 80)
print("오류 유형 분포")
print("=" * 80)

print(
    error_summary
    .to_string(
        index=False
    )
)


# ============================================================
# 13. FN 상세 분석
#
# 실제 불량인데 모델이 놓친 제품
# ============================================================

fn_cases = (
    cn7_analysis[
        cn7_analysis[
            "Result_Type"
        ] == "FN"
    ]
    .sort_values(
        "Risk_Probability",
        ascending=False
    )
)


print("\n" + "=" * 80)
print("FALSE NEGATIVE - 실제 불량인데 놓친 사례")
print("=" * 80)

print(
    fn_cases[
        [
            "TimeStamp",
            "Side_RH",
            "Risk_Score",
            "Risk_STD",
            "Positive_Votes",
            "Risk_Level"
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 14. FP 상세 분석
#
# 실제 정상인데 불량으로 판단한 제품
# ============================================================

fp_cases = (
    cn7_analysis[
        cn7_analysis[
            "Result_Type"
        ] == "FP"
    ]
    .sort_values(
        "Risk_Probability",
        ascending=False
    )
)


print("\n" + "=" * 80)
print("FALSE POSITIVE - 실제 정상인데 불량으로 판단한 사례")
print("=" * 80)

print(
    fp_cases[
        [
            "TimeStamp",
            "Side_RH",
            "Risk_Score",
            "Risk_STD",
            "Positive_Votes",
            "Risk_Level"
        ]
    ]
    .head(20)
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 15. FN vs TP Feature 차이 분석
#
# 어떤 Feature에서 놓친 불량과 잡은 불량의 차이가 큰지 확인
# ============================================================

feature_cols = (
    X_final.columns.tolist()
)

tp_data = (
    cn7_analysis[
        cn7_analysis[
            "Result_Type"
        ] == "TP"
    ]
)

fn_data = (
    cn7_analysis[
        cn7_analysis[
            "Result_Type"
        ] == "FN"
    ]
)


fn_feature_analysis = []

if len(fn_data) > 0:

    for feature in feature_cols:

        tp_median = (
            tp_data[
                feature
            ].median()
        )

        fn_median = (
            fn_data[
                feature
            ].median()
        )

        fn_feature_analysis.append({

            "Feature":
                feature,

            "TP_Median":
                tp_median,

            "FN_Median":
                fn_median,

            "Difference":
                fn_median
                - tp_median,

            "Abs_Difference":
                abs(
                    fn_median
                    - tp_median
                )
        })


fn_feature_analysis = pd.DataFrame(
    fn_feature_analysis
)

fn_feature_analysis = (
    fn_feature_analysis
    .sort_values(
        "Abs_Difference",
        ascending=False
    )
)


print("\n" + "=" * 80)
print("FN vs TP 주요 Feature 차이 TOP 15")
print("=" * 80)

print(
    fn_feature_analysis
    .head(15)
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 16. FP vs TN Feature 차이 분석
# ============================================================

fp_data = (
    cn7_analysis[
        cn7_analysis[
            "Result_Type"
        ] == "FP"
    ]
)

tn_data = (
    cn7_analysis[
        cn7_analysis[
            "Result_Type"
        ] == "TN"
    ]
)


fp_feature_analysis = []


if len(fp_data) > 0:

    for feature in feature_cols:

        tn_median = (
            tn_data[
                feature
            ].median()
        )

        fp_median = (
            fp_data[
                feature
            ].median()
        )

        fp_feature_analysis.append({

            "Feature":
                feature,

            "TN_Median":
                tn_median,

            "FP_Median":
                fp_median,

            "Difference":
                fp_median
                - tn_median,

            "Abs_Difference":
                abs(
                    fp_median
                    - tn_median
                )
        })


fp_feature_analysis = pd.DataFrame(
    fp_feature_analysis
)

fp_feature_analysis = (
    fp_feature_analysis
    .sort_values(
        "Abs_Difference",
        ascending=False
    )
)


print("\n" + "=" * 80)
print("FP vs TN 주요 Feature 차이 TOP 15")
print("=" * 80)

print(
    fp_feature_analysis
    .head(15)
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 17. 검사 우선순위 테이블 생성
# ============================================================

inspection_priority = (
    cn7_analysis[
        [
            "Inspection_Rank",
            "TimeStamp",
            "Side_RH",
            "Risk_Score",
            "Risk_Level",
            "Prediction_Stability",
            "Inspection_Action",
            "Actual"
        ]
    ]
    .sort_values(
        "Inspection_Rank"
    )
)


print("\n" + "=" * 80)
print("검사 우선순위 TOP 30")
print("=" * 80)

print(
    inspection_priority
    .head(30)
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 18. 결과 저장
# ============================================================

cn7_analysis.to_csv(
    "CN7_quality_risk_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

inspection_priority.to_csv(
    "CN7_inspection_priority.csv",
    index=False,
    encoding="utf-8-sig"
)

side_performance.to_csv(
    "CN7_side_performance.csv",
    index=False,
    encoding="utf-8-sig"
)

risk_level_summary.to_csv(
    "CN7_risk_level_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

stability_summary.to_csv(
    "CN7_prediction_stability.csv",
    index=False,
    encoding="utf-8-sig"
)

fn_feature_analysis.to_csv(
    "CN7_FN_feature_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

fp_feature_analysis.to_csv(
    "CN7_FP_feature_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n저장 완료")
print("- CN7_quality_risk_analysis.csv")
print("- CN7_inspection_priority.csv")
print("- CN7_side_performance.csv")
print("- CN7_risk_level_summary.csv")
print("- CN7_prediction_stability.csv")
print("- CN7_FN_feature_analysis.csv")
print("- CN7_FP_feature_analysis.csv")

Risk + Uncertainty 검사 정책 검증 코드

In [ ]:
# ============================================================
# CN7 Risk + Uncertainty 기반 검사 우선순위 정책
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. 기존 정책
# ------------------------------------------------------------

cn7_analysis["Risk_Only_Flag"] = (
    cn7_analysis["Risk_Probability"] >= 0.060
).astype(int)


# ------------------------------------------------------------
# 2. 개선 정책
#
# 조건 A: Risk Probability >= 0.060
# OR
# 조건 B: Seed 간 예측이 불안정
# ------------------------------------------------------------

cn7_analysis["Risk_Uncertainty_Flag"] = (
    (
        cn7_analysis["Risk_Probability"] >= 0.060
    )
    |
    (
        cn7_analysis["Prediction_Stability"] == "Unstable"
    )
).astype(int)


# ------------------------------------------------------------
# 3. 정책 성능 비교 함수
# ------------------------------------------------------------

def evaluate_policy(
    y_true,
    pred,
    name
):

    precision = precision_score(
        y_true,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred
    ).ravel()

    flag_rate = (
        pred.sum()
        / len(pred)
        * 100
    )

    return {
        "Policy": name,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Flag_Count": pred.sum(),
        "Flag_Rate_pct": flag_rate
    }


policy_results = []

policy_results.append(
    evaluate_policy(
        cn7_analysis["Actual"],
        cn7_analysis["Risk_Only_Flag"],
        "Risk Only"
    )
)

policy_results.append(
    evaluate_policy(
        cn7_analysis["Actual"],
        cn7_analysis["Risk_Uncertainty_Flag"],
        "Risk + Uncertainty"
    )
)

policy_comparison = pd.DataFrame(
    policy_results
)

print(
    policy_comparison
    .round(4)
    .to_string(index=False)
)

# 검사순위 조정

In [ ]:
def inspection_priority_v2(row):

    # 가장 높은 위험
    if row["Risk_Probability"] >= 0.470:
        return "P1_CRITICAL"

    # 위험도 기준 검사
    elif row["Risk_Probability"] >= 0.060:
        return "P2_HIGH"

    # 위험도는 낮지만 모델이 확신하지 못함
    elif row["Prediction_Stability"] == "Unstable":
        return "P3_UNCERTAIN"

    else:
        return "P4_NORMAL"


cn7_analysis[
    "Inspection_Priority_V2"
] = cn7_analysis.apply(
    inspection_priority_v2,
    axis=1
)

In [ ]:
priority_order = {
    "P1_CRITICAL": 1,
    "P2_HIGH": 2,
    "P3_UNCERTAIN": 3,
    "P4_NORMAL": 4
}

cn7_analysis[
    "Priority_Order"
] = (
    cn7_analysis[
        "Inspection_Priority_V2"
    ].map(
        priority_order
    )
)

inspection_priority_v2 = (
    cn7_analysis
    .sort_values(
        [
            "Priority_Order",
            "Risk_Probability"
        ],
        ascending=[
            True,
            False
        ]
    )
    .copy()
)

inspection_priority_v2[
    "Inspection_Rank_V2"
] = np.arange(
    1,
    len(inspection_priority_v2) + 1
)

In [ ]:
# ============================================================
# CN7 Risk Only vs Risk + Uncertainty 최종 비교
# ============================================================

import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. 필요한 컬럼 확인
# ------------------------------------------------------------

print("cn7_analysis shape:", cn7_analysis.shape)

print(
    "필요 컬럼:",
    [
        "Actual",
        "Risk_Probability",
        "Prediction_Stability"
    ]
)


# ------------------------------------------------------------
# 2. 기존 Risk Only 정책
# ------------------------------------------------------------

cn7_analysis["Risk_Only_Flag"] = (
    cn7_analysis["Risk_Probability"] >= 0.060
).astype(int)


# ------------------------------------------------------------
# 3. Risk + Uncertainty 정책
#
# Risk >= 0.06
# 또는
# 5개 Seed 중 예측이 불안정한 제품
# ------------------------------------------------------------

cn7_analysis["Risk_Uncertainty_Flag"] = (
    (
        cn7_analysis["Risk_Probability"] >= 0.060
    )
    |
    (
        cn7_analysis["Prediction_Stability"] == "Unstable"
    )
).astype(int)


# ------------------------------------------------------------
# 4. 평가 함수
# ------------------------------------------------------------

def evaluate_policy(name, pred):

    y_true = cn7_analysis["Actual"]

    precision = precision_score(
        y_true,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred
    ).ravel()

    flag_count = int(pred.sum())

    flag_rate = (
        flag_count
        / len(pred)
        * 100
    )

    return {
        "Policy": name,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Flag_Count": flag_count,
        "Flag_Rate_pct": flag_rate
    }


# ------------------------------------------------------------
# 5. 비교
# ------------------------------------------------------------

results = [
    evaluate_policy(
        "Risk Only",
        cn7_analysis["Risk_Only_Flag"]
    ),

    evaluate_policy(
        "Risk + Uncertainty",
        cn7_analysis["Risk_Uncertainty_Flag"]
    )
]

policy_comparison = pd.DataFrame(results)


# ------------------------------------------------------------
# 6. 무조건 출력
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CN7 INSPECTION POLICY COMPARISON")
print("=" * 100)

print(
    policy_comparison
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 7. 추가 검사된 제품 확인
# ------------------------------------------------------------

extra_inspection = cn7_analysis[
    (
        cn7_analysis["Risk_Only_Flag"] == 0
    )
    &
    (
        cn7_analysis["Risk_Uncertainty_Flag"] == 1
    )
].copy()


print("\n" + "=" * 100)
print("Risk Only에서는 빠졌지만 Uncertainty 때문에 추가 검사되는 제품")
print("=" * 100)

print(
    extra_inspection[
        [
            "TimeStamp",
            "Side_RH",
            "Actual",
            "Risk_Score",
            "Risk_STD",
            "Positive_Votes",
            "Prediction_Stability"
        ]
    ]
    .round(4)
    .to_string(index=False)
)

# CN7 최종 실패사례 분석 + 검사 우선순위 생성

In [ ]:
# ============================================================
# CN7 FINAL
# 실패 사례(FN) 심층 분석 + Risk/Uncertainty 기반 검사 우선순위
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.metrics import pairwise_distances


# ============================================================
# 0. 기본 설정
# ============================================================

RISK_CRITICAL = 0.470
RISK_HIGH = 0.060

FEATURE_COLS = X_final.columns.tolist()


# ============================================================
# 1. 최종 검사 정책 생성
#
# Risk >= 0.06
# OR
# Prediction_Stability == Unstable
# ============================================================

cn7_analysis["Final_Inspection_Flag"] = (
    (
        cn7_analysis["Risk_Probability"] >= RISK_HIGH
    )
    |
    (
        cn7_analysis["Prediction_Stability"] == "Unstable"
    )
).astype(int)


# ============================================================
# 2. 최종 정책 기준 TP / FP / FN / TN 재생성
# ============================================================

conditions = [
    (
        (cn7_analysis["Actual"] == 1)
        & (cn7_analysis["Final_Inspection_Flag"] == 1)
    ),
    (
        (cn7_analysis["Actual"] == 0)
        & (cn7_analysis["Final_Inspection_Flag"] == 1)
    ),
    (
        (cn7_analysis["Actual"] == 1)
        & (cn7_analysis["Final_Inspection_Flag"] == 0)
    ),
    (
        (cn7_analysis["Actual"] == 0)
        & (cn7_analysis["Final_Inspection_Flag"] == 0)
    )
]

labels = [
    "TP",
    "FP",
    "FN",
    "TN"
]

cn7_analysis["Final_Result_Type"] = np.select(
    conditions,
    labels,
    default="Unknown"
)


# ============================================================
# 3. 최종 정책 성능 출력
# ============================================================

y_true = cn7_analysis["Actual"]
y_pred = cn7_analysis["Final_Inspection_Flag"]

precision_final = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall_final = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1_final = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

tn, fp, fn, tp = confusion_matrix(
    y_true,
    y_pred
).ravel()

flag_count = int(
    y_pred.sum()
)

flag_rate = (
    flag_count
    / len(y_pred)
    * 100
)


print("\n" + "=" * 100)
print("CN7 FINAL INSPECTION POLICY PERFORMANCE")
print("=" * 100)

print(f"Precision       : {precision_final:.4f}")
print(f"Recall          : {recall_final:.4f}")
print(f"F1              : {f1_final:.4f}")

print()

print(f"TP              : {tp}")
print(f"FP              : {fp}")
print(f"FN              : {fn}")
print(f"TN              : {tn}")

print()

print(f"Inspection Count: {flag_count}")
print(f"Inspection Rate : {flag_rate:.4f}%")


# ============================================================
# 4. 최종 남은 FN 추출
# ============================================================

final_fn = (
    cn7_analysis[
        cn7_analysis[
            "Final_Result_Type"
        ] == "FN"
    ]
    .copy()
)


print("\n" + "=" * 100)
print("FINAL FALSE NEGATIVE CASE")
print("=" * 100)

if len(final_fn) == 0:

    print("남은 FN 없음")

else:

    fn_print_cols = [
        "TimeStamp",
        "Side_RH",
        "Actual",
        "Risk_Score",
        "Risk_STD",
        "Positive_Votes",
        "Prediction_Stability"
    ]

    print(
        final_fn[
            fn_print_cols
        ].to_string(
            index=False
        )
    )


# ============================================================
# 5. 남은 FN vs TP / TN 비교
#
# FN Feature 값이
# - 잡힌 불량(TP)
# - 정상(TN)
# 중 어디에 더 가까운지 확인
# ============================================================

tp_final = (
    cn7_analysis[
        cn7_analysis[
            "Final_Result_Type"
        ] == "TP"
    ]
)

tn_final = (
    cn7_analysis[
        cn7_analysis[
            "Final_Result_Type"
        ] == "TN"
    ]
)


fn_comparison = []

if len(final_fn) > 0:

    fn_row = final_fn.iloc[0]

    for feature in FEATURE_COLS:

        fn_value = fn_row[feature]

        tp_median = (
            tp_final[
                feature
            ].median()
        )

        tn_median = (
            tn_final[
                feature
            ].median()
        )

        overall_std = (
            cn7_analysis[
                feature
            ].std()
        )

        if overall_std == 0 or pd.isna(overall_std):

            dist_tp = np.nan
            dist_tn = np.nan

        else:

            dist_tp = (
                abs(
                    fn_value
                    - tp_median
                )
                / overall_std
            )

            dist_tn = (
                abs(
                    fn_value
                    - tn_median
                )
                / overall_std
            )

        fn_comparison.append({
            "Feature":
                feature,

            "FN_Value":
                fn_value,

            "TP_Median":
                tp_median,

            "TN_Median":
                tn_median,

            "Distance_to_TP":
                dist_tp,

            "Distance_to_TN":
                dist_tn,

            "Closer_To":
                (
                    "TP"
                    if dist_tp < dist_tn
                    else "TN"
                )
        })


fn_comparison_df = pd.DataFrame(
    fn_comparison
)


# ============================================================
# 6. FN이 정상 패턴에 더 가까운 Feature 확인
# ============================================================

fn_normal_like = (
    fn_comparison_df[
        fn_comparison_df[
            "Closer_To"
        ] == "TN"
    ]
    .copy()
)


fn_normal_like[
    "TN_Advantage"
] = (
    fn_normal_like[
        "Distance_to_TP"
    ]
    - fn_normal_like[
        "Distance_to_TN"
    ]
)


fn_normal_like = (
    fn_normal_like
    .sort_values(
        "TN_Advantage",
        ascending=False
    )
)


print("\n" + "=" * 100)
print("남은 FN이 정상(TN) 패턴에 더 가까운 Feature TOP 15")
print("=" * 100)

print(
    fn_normal_like[
        [
            "Feature",
            "FN_Value",
            "TP_Median",
            "TN_Median",
            "Distance_to_TP",
            "Distance_to_TN"
        ]
    ]
    .head(15)
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 7. FN Feature Percentile 분석
#
# 전체 데이터에서 해당 값이 어느 위치인지 확인
# ============================================================

fn_percentiles = []

if len(final_fn) > 0:

    fn_row = final_fn.iloc[0]

    for feature in FEATURE_COLS:

        value = (
            fn_row[
                feature
            ]
        )

        percentile = (
            cn7_analysis[
                feature
            ]
            .rank(
                pct=True
            )
            .loc[
                fn_row.name
            ]
            * 100
        )

        fn_percentiles.append({
            "Feature":
                feature,

            "FN_Value":
                value,

            "Percentile":
                percentile,

            "Extreme_Score":
                abs(
                    percentile - 50
                )
        })


fn_percentile_df = pd.DataFrame(
    fn_percentiles
)


fn_percentile_df = (
    fn_percentile_df
    .sort_values(
        "Extreme_Score",
        ascending=False
    )
)


print("\n" + "=" * 100)
print("남은 FN Feature 극단값 TOP 15")
print("=" * 100)

print(
    fn_percentile_df[
        [
            "Feature",
            "FN_Value",
            "Percentile"
        ]
    ]
    .head(15)
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 8. Nearest Neighbor 분석
#
# 남은 FN이 실제로 어떤 제품들과
# Feature 공간에서 가까운지 확인
#
# 같은 TimeStamp는 제외
# ============================================================

if len(final_fn) > 0:

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        cn7_analysis[
            FEATURE_COLS
        ]
    )

    fn_index = (
        final_fn.index[0]
    )

    fn_position = (
        cn7_analysis
        .index
        .get_loc(
            fn_index
        )
    )

    fn_vector = (
        X_scaled[
            fn_position
        ]
        .reshape(
            1,
            -1
        )
    )

    distances = (
        pairwise_distances(
            fn_vector,
            X_scaled,
            metric="euclidean"
        )
        .flatten()
    )

    neighbor_df = (
        cn7_analysis[
            [
                "TimeStamp",
                "Side_RH",
                "Actual",
                "Risk_Score",
                "Prediction_Stability"
            ]
        ]
        .copy()
    )

    neighbor_df[
        "Distance"
    ] = distances


    # 자기 자신 제외
    neighbor_df = (
        neighbor_df[
            neighbor_df.index
            != fn_index
        ]
    )


    # 같은 생산 Cycle 제외
    fn_timestamp = (
        final_fn.iloc[0][
            "TimeStamp"
        ]
    )

    neighbor_df = (
        neighbor_df[
            neighbor_df[
                "TimeStamp"
            ] != fn_timestamp
        ]
    )


    nearest_10 = (
        neighbor_df
        .sort_values(
            "Distance"
        )
        .head(10)
    )


    print("\n" + "=" * 100)
    print("남은 FN과 가장 유사한 다른 생산품 TOP 10")
    print("=" * 100)

    print(
        nearest_10
        .round(
            {
                "Risk_Score": 4,
                "Distance": 4
            }
        )
        .to_string(
            index=False
        )
    )


    print("\nNearest 10 Actual 분포")

    print(
        nearest_10[
            "Actual"
        ]
        .value_counts()
        .rename(
            index={
                0:
                    "Normal",
                1:
                    "Defect"
            }
        )
        .to_string()
    )


# ============================================================
# 9. 최종 검사 우선순위 P1~P4
# ============================================================

def final_priority(row):

    # ----------------------------------
    # P1
    # 매우 높은 위험
    # ----------------------------------

    if (
        row[
            "Risk_Probability"
        ]
        >= RISK_CRITICAL
    ):

        return "P1_CRITICAL"


    # ----------------------------------
    # P2
    # 일반적인 고위험
    # ----------------------------------

    elif (
        row[
            "Risk_Probability"
        ]
        >= RISK_HIGH
    ):

        return "P2_HIGH"


    # ----------------------------------
    # P3
    # 위험도 자체는 낮지만
    # Seed간 판단 불일치
    # ----------------------------------

    elif (
        row[
            "Prediction_Stability"
        ]
        == "Unstable"
    ):

        return "P3_UNCERTAIN"


    # ----------------------------------
    # P4
    # 저위험 + 안정적 예측
    # ----------------------------------

    else:

        return "P4_NORMAL"


cn7_analysis[
    "Inspection_Priority"
] = (
    cn7_analysis
    .apply(
        final_priority,
        axis=1
    )
)


# ============================================================
# 10. Priority 설명 추가
# ============================================================

priority_reason = {

    "P1_CRITICAL":
        "불량 위험도 매우 높음",

    "P2_HIGH":
        "불량 위험도 기준 초과",

    "P3_UNCERTAIN":
        "위험도는 낮지만 모델 판단 불안정",

    "P4_NORMAL":
        "저위험 및 예측 안정"
}


priority_action = {

    "P1_CRITICAL":
        "즉시 검사",

    "P2_HIGH":
        "우선 검사",

    "P3_UNCERTAIN":
        "불확실성 추가 검사",

    "P4_NORMAL":
        "일반 검사"
}


cn7_analysis[
    "Priority_Reason"
] = (
    cn7_analysis[
        "Inspection_Priority"
    ]
    .map(
        priority_reason
    )
)


cn7_analysis[
    "Inspection_Action_Final"
] = (
    cn7_analysis[
        "Inspection_Priority"
    ]
    .map(
        priority_action
    )
)


# ============================================================
# 11. 최종 순위 생성
#
# P1 → P2 → P3 → P4
#
# P1/P2:
# Risk 높은 순
#
# P3:
# Seed 변동성 높은 순
# ============================================================

priority_order = {

    "P1_CRITICAL": 1,
    "P2_HIGH": 2,
    "P3_UNCERTAIN": 3,
    "P4_NORMAL": 4
}


cn7_analysis[
    "Priority_Order"
] = (
    cn7_analysis[
        "Inspection_Priority"
    ]
    .map(
        priority_order
    )
)


final_priority_df = (
    cn7_analysis
    .sort_values(
        [
            "Priority_Order",
            "Risk_Probability",
            "Risk_STD"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .copy()
)


final_priority_df[
    "Inspection_Rank"
] = (
    np.arange(
        1,
        len(
            final_priority_df
        ) + 1
    )
)


# ============================================================
# 12. P1~P4 요약
# ============================================================

priority_summary = (
    final_priority_df
    .groupby(
        "Inspection_Priority",
        observed=True
    )
    .agg(

        Count=(
            "Actual",
            "size"
        ),

        Actual_Defects=(
            "Actual",
            "sum"
        ),

        Actual_Defect_Rate=(
            "Actual",
            "mean"
        ),

        Avg_Risk_Score=(
            "Risk_Score",
            "mean"
        )
    )
    .reset_index()
)


priority_summary[
    "Inspection_Rate_pct"
] = (
    priority_summary[
        "Count"
    ]
    / len(
        cn7_analysis
    )
    * 100
)


print("\n" + "=" * 100)
print("FINAL PRIORITY SUMMARY")
print("=" * 100)

print(
    priority_summary
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 13. 검사 우선순위 TOP 30 출력
# ============================================================

print("\n" + "=" * 100)
print("FINAL INSPECTION PRIORITY TOP 30")
print("=" * 100)

print(
    final_priority_df[
        [
            "Inspection_Rank",
            "TimeStamp",
            "Side_RH",
            "Risk_Score",
            "Risk_STD",
            "Positive_Votes",
            "Prediction_Stability",
            "Inspection_Priority",
            "Priority_Reason",
            "Inspection_Action_Final",
            "Actual"
        ]
    ]
    .head(30)
    .round(
        {
            "Risk_Score": 4,
            "Risk_STD": 4
        }
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 14. CSV 저장
# ============================================================

final_priority_df.to_csv(
    "CN7_final_inspection_priority.csv",
    index=False,
    encoding="utf-8-sig"
)


priority_summary.to_csv(
    "CN7_final_priority_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


fn_comparison_df.to_csv(
    "CN7_final_FN_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)


fn_percentile_df.to_csv(
    "CN7_final_FN_percentile.csv",
    index=False,
    encoding="utf-8-sig"
)


if len(final_fn) > 0:

    nearest_10.to_csv(
        "CN7_final_FN_nearest_neighbors.csv",
        index=False,
        encoding="utf-8-sig"
    )


print("\n" + "=" * 100)
print("저장 완료")
print("=" * 100)

print(
    "CN7_final_inspection_priority.csv"
)

print(
    "CN7_final_priority_summary.csv"
)

print(
    "CN7_final_FN_analysis.csv"
)

print(
    "CN7_final_FN_percentile.csv"
)

if len(final_fn) > 0:

    print(
        "CN7_final_FN_nearest_neighbors.csv"
    )

# CN7 Failure Episode 탐색

In [ ]:
# ============================================================
# CN7 Side_RH Only Sanity Check
# 목적:
# Side_RH 하나만으로 모델 성능이 어느 정도 나오는지 확인
# ============================================================

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)


# ============================================================
# 1. Side_RH 하나만 사용
# ============================================================

X_side_only = X_final[
    ["Side_RH"]
].copy()

print("Side-only Feature 수 :", X_side_only.shape[1])
print("Feature :", X_side_only.columns.tolist())


# ============================================================
# 2. 기존 최적 LightGBM 파라미터 그대로 사용
# ============================================================

best_params_cn7 = {
    "n_estimators": 200,
    "learning_rate": 0.10,
    "num_leaves": 15,
    "max_depth": 7,
    "min_child_samples": 10,
    "subsample": 1.0,
    "colsample_bytree": 0.70,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0
}


# ============================================================
# 3. 반복 검증 설정
# ============================================================

SEEDS = [
    42,
    100,
    2026,
    777,
    1234
]

N_SPLITS = 4

oof_side_only = {}
pr_auc_side_only = []


# ============================================================
# 4. 5 Seeds × 4 Fold Group CV
# ============================================================

for seed in SEEDS:

    sgkf = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    oof_proba = np.zeros(
        len(X_side_only)
    )

    for train_idx, val_idx in sgkf.split(
        X_side_only,
        y_final,
        groups=groups_final
    ):

        X_train = X_side_only.iloc[
            train_idx
        ].copy()

        X_val = X_side_only.iloc[
            val_idx
        ].copy()

        y_train = y_final.iloc[
            train_idx
        ].copy()

        y_val = y_final.iloc[
            val_idx
        ].copy()


        model = LGBMClassifier(
            **best_params_cn7,
            class_weight="balanced",
            random_state=42,
            verbose=-1
        )


        model.fit(
            X_train,
            y_train
        )


        val_proba = (
            model
            .predict_proba(
                X_val
            )[:, 1]
        )


        oof_proba[
            val_idx
        ] = val_proba


    oof_side_only[
        seed
    ] = oof_proba


    seed_pr_auc = (
        average_precision_score(
            y_final,
            oof_proba
        )
    )

    pr_auc_side_only.append(
        seed_pr_auc
    )


    print(
        f"Seed {seed} | "
        f"PR-AUC: {seed_pr_auc:.4f}"
    )


# ============================================================
# 5. Side-only PR-AUC 요약
# ============================================================

side_only_mean = np.mean(
    pr_auc_side_only
)

side_only_std = np.std(
    pr_auc_side_only
)


print("\n" + "=" * 60)

print("Side_RH ONLY PR-AUC")

print("=" * 60)

print(
    f"Mean : {side_only_mean:.4f}"
)

print(
    f"Std  : {side_only_std:.4f}"
)

print(
    f"Min  : {np.min(pr_auc_side_only):.4f}"
)

print(
    f"Max  : {np.max(pr_auc_side_only):.4f}"
)


# ============================================================
# 6. Side-only Threshold 탐색
# ============================================================

thresholds = np.arange(
    0.005,
    0.501,
    0.005
)

threshold_side_only_results = []


for threshold in thresholds:

    seed_metrics = []


    for seed in SEEDS:

        proba = (
            oof_side_only[
                seed
            ]
        )


        pred = (
            proba >= threshold
        ).astype(int)


        precision = (
            precision_score(
                y_final,
                pred,
                zero_division=0
            )
        )


        recall = (
            recall_score(
                y_final,
                pred,
                zero_division=0
            )
        )


        f1 = (
            f1_score(
                y_final,
                pred,
                zero_division=0
            )
        )


        f2 = (
            fbeta_score(
                y_final,
                pred,
                beta=2,
                zero_division=0
            )
        )


        tn, fp, fn, tp = (
            confusion_matrix(
                y_final,
                pred
            ).ravel()
        )


        seed_metrics.append({
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "F2": f2,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn
        })


    temp = pd.DataFrame(
        seed_metrics
    )


    threshold_side_only_results.append({

        "Threshold":
            round(
                threshold,
                3
            ),

        "Precision_mean":
            temp[
                "Precision"
            ].mean(),

        "Recall_mean":
            temp[
                "Recall"
            ].mean(),

        "F1_mean":
            temp[
                "F1"
            ].mean(),

        "F2_mean":
            temp[
                "F2"
            ].mean(),

        "TP_mean":
            temp[
                "TP"
            ].mean(),

        "FP_mean":
            temp[
                "FP"
            ].mean(),

        "FN_mean":
            temp[
                "FN"
            ].mean()
    })


threshold_side_only_df = pd.DataFrame(
    threshold_side_only_results
)


# ============================================================
# 7. Side-only F1 최고 Threshold
# ============================================================

best_f1_side_only = (
    threshold_side_only_df
    .loc[
        threshold_side_only_df[
            "F1_mean"
        ].idxmax()
    ]
)


print(
    "\n[F1 기준 Side-only 최적 Threshold]"
)

print(
    best_f1_side_only
    .round(4)
    .to_string()
)


# ============================================================
# 8. 최종 비교
# ============================================================

comparison_final = pd.DataFrame({

    "Model": [
        "Full Model",
        "Without Side_RH",
        "Side_RH Only"
    ],

    "Features": [
        47,
        46,
        1
    ],

    "PR_AUC": [
        0.8157,
        0.6509,
        side_only_mean
    ],

    "Precision": [
        0.7508,
        0.6093,
        best_f1_side_only[
            "Precision_mean"
        ]
    ],

    "Recall": [
        0.7647,
        0.5647,
        best_f1_side_only[
            "Recall_mean"
        ]
    ],

    "F1": [
        0.7558,
        0.5843,
        best_f1_side_only[
            "F1_mean"
        ]
    ],

    "F2": [
        0.7606,
        0.5720,
        best_f1_side_only[
            "F2_mean"
        ]
    ]
})


print(
    "\n" + "=" * 90
)

print(
    "CN7 SIDE SANITY CHECK FINAL"
)

print(
    "=" * 90
)


print(
    comparison_final
    .round(4)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# CN7 Failure Episode 탐색
#
# 목적
# 1. TimeStamp 단위 생산 Cycle 생성
# 2. Cycle별 불량 발생 여부 확인
# 3. 불량 Cycle 간 시간 간격 계산
# 4. 시간축에서 불량이 몰려 있는 구간 확인
# 5. Episode 정의 후보(5/10/30/60분) 비교
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. 사용할 원본 데이터 확인
# ============================================================

# cn7_work에 TimeStamp / PassOrFail이 있는지 확인
print("cn7_work shape:", cn7_work.shape)

print(
    "\n필요 컬럼 존재 여부:"
)

for col in [
    "TimeStamp",
    "PassOrFail",
    "Side_RH"
]:
    print(
        f"{col}:",
        col in cn7_work.columns
    )


# ============================================================
# 2. TimeStamp 단위 Cycle 데이터 생성
#
# 동일 TimeStamp에 LH/RH가 있으므로
# Cycle 하나라도 불량이면 해당 Cycle을 불량 Cycle로 정의
# ============================================================

cycle_df = (
    cn7_work
    .groupby(
        "TimeStamp",
        as_index=False
    )
    .agg(
        Cycle_Defect=(
            "PassOrFail",
            "max"
        ),

        Defect_Count=(
            "PassOrFail",
            "sum"
        ),

        Row_Count=(
            "PassOrFail",
            "size"
        )
    )
    .sort_values(
        "TimeStamp"
    )
    .reset_index(drop=True)
)


# Cycle 순서
cycle_df[
    "Cycle_Index"
] = np.arange(
    len(cycle_df)
)


print(
    "\n전체 Cycle 수:",
    len(cycle_df)
)

print(
    "불량 Cycle 수:",
    int(
        cycle_df[
            "Cycle_Defect"
        ].sum()
    )
)


# ============================================================
# 3. 불량 Cycle만 추출
# ============================================================

defect_cycles = (
    cycle_df[
        cycle_df[
            "Cycle_Defect"
        ] == 1
    ]
    .copy()
    .reset_index(drop=True)
)


# 이전 불량 Cycle과의 시간 차이
defect_cycles[
    "Gap_From_Previous_Defect_Min"
] = (
    defect_cycles[
        "TimeStamp"
    ]
    .diff()
    .dt
    .total_seconds()
    / 60
)


# 이전 불량과 Cycle 순서 차이
defect_cycles[
    "Cycle_Gap"
] = (
    defect_cycles[
        "Cycle_Index"
    ]
    .diff()
)


print(
    "\n"
    + "=" * 100
)

print(
    "CN7 DEFECT CYCLE TIMELINE"
)

print(
    "=" * 100
)

print(
    defect_cycles[
        [
            "Cycle_Index",
            "TimeStamp",
            "Defect_Count",
            "Gap_From_Previous_Defect_Min",
            "Cycle_Gap"
        ]
    ]
    .round(
        {
            "Gap_From_Previous_Defect_Min": 2
        }
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 4. 불량 Cycle 시간 간격 통계
# ============================================================

gap_series = (
    defect_cycles[
        "Gap_From_Previous_Defect_Min"
    ]
    .dropna()
)


print(
    "\n"
    + "=" * 100
)

print(
    "DEFECT GAP SUMMARY"
)

print(
    "=" * 100
)

print(
    gap_series
    .describe()
    .round(2)
)


# ============================================================
# 5. Episode 정의 함수
#
# 이전 불량과 일정 시간 이상 떨어지면
# 새로운 failure episode로 정의
# ============================================================

def make_episode(
    defect_data,
    gap_threshold_min
):

    temp = (
        defect_data
        .copy()
    )

    new_episode = (
        temp[
            "Gap_From_Previous_Defect_Min"
        ]
        .isna()
        |
        (
            temp[
                "Gap_From_Previous_Defect_Min"
            ]
            > gap_threshold_min
        )
    )

    temp[
        "Episode_ID"
    ] = (
        new_episode
        .cumsum()
    )

    return temp


# ============================================================
# 6. 후보 Threshold 비교
# ============================================================

candidate_thresholds = [
    5,
    10,
    30,
    60
]

episode_summary_list = []


for threshold in candidate_thresholds:

    temp = make_episode(
        defect_cycles,
        threshold
    )

    summary = (
        temp
        .groupby(
            "Episode_ID"
        )
        .agg(
            Start_Time=(
                "TimeStamp",
                "min"
            ),

            End_Time=(
                "TimeStamp",
                "max"
            ),

            Defect_Cycles=(
                "Cycle_Defect",
                "size"
            ),

            Start_Cycle=(
                "Cycle_Index",
                "min"
            ),

            End_Cycle=(
                "Cycle_Index",
                "max"
            )
        )
        .reset_index()
    )

    episode_summary_list.append({
        "Gap_Threshold_Min":
            threshold,

        "Episode_Count":
            len(summary),

        "Max_Defects_In_Episode":
            summary[
                "Defect_Cycles"
            ].max(),

        "Mean_Defects_Per_Episode":
            summary[
                "Defect_Cycles"
            ].mean()
    })


episode_threshold_summary = pd.DataFrame(
    episode_summary_list
)


print(
    "\n"
    + "=" * 100
)

print(
    "FAILURE EPISODE THRESHOLD COMPARISON"
)

print(
    "=" * 100
)

print(
    episode_threshold_summary
    .round(2)
    .to_string(
        index=False
    )
)


# ============================================================
# 7. 불량 Cycle 시간축 시각화
# ============================================================

plt.figure(
    figsize=(
        14,
        4
    )
)


plt.scatter(
    cycle_df[
        "TimeStamp"
    ],
    cycle_df[
        "Cycle_Defect"
    ],
    s=20
)


# 불량만 크게 표시
plt.scatter(
    defect_cycles[
        "TimeStamp"
    ],
    np.ones(
        len(defect_cycles)
    ),
    s=80
)


plt.yticks(
    [
        0,
        1
    ],
    [
        "Normal",
        "Defect"
    ]
)


plt.xlabel(
    "Production Time"
)

plt.ylabel(
    "Cycle Status"
)

plt.title(
    "CN7 Defect Occurrence Timeline"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# 8. 불량 Cycle 간 Gap 시각화
# ============================================================

plt.figure(
    figsize=(
        12,
        4
    )
)


plt.plot(
    defect_cycles[
        "TimeStamp"
    ],
    defect_cycles[
        "Gap_From_Previous_Defect_Min"
    ],
    marker="o"
)


plt.xlabel(
    "Defect Time"
)

plt.ylabel(
    "Minutes From Previous Defect"
)

plt.title(
    "CN7 Time Gap Between Defect Cycles"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CN7 Chronological Phase 확인
# ============================================================

temp = cn7_work[
    [
        "TimeStamp",
        "PassOrFail",
        "Side_RH"
    ]
].copy()

temp["Date"] = (
    temp["TimeStamp"]
    .dt.date
)

date_summary = (
    temp
    .groupby("Date")
    .agg(
        Rows=("PassOrFail", "size"),
        Defects=("PassOrFail", "sum"),
        Normal=("PassOrFail", lambda x: (x == 0).sum()),
        Start_Time=("TimeStamp", "min"),
        End_Time=("TimeStamp", "max")
    )
    .reset_index()
)

print(
    "\n"
    + "=" * 100
)

print(
    "CN7 DATE SUMMARY"
)

print(
    "=" * 100
)

print(
    date_summary.to_string(
        index=False
    )
)


# ============================================================
# 마지막 불량 이후 데이터 규모
# ============================================================

last_defect_time = (
    cn7_work
    .loc[
        cn7_work["PassOrFail"] == 1,
        "TimeStamp"
    ]
    .max()
)

future_after_failure = (
    cn7_work[
        cn7_work["TimeStamp"]
        > last_defect_time
    ]
    .copy()
)


print(
    "\nLast Defect Time:",
    last_defect_time
)

print(
    "Rows after last defect:",
    len(future_after_failure)
)

print(
    "Defects after last defect:",
    int(
        future_after_failure[
            "PassOrFail"
        ].sum()
    )
)

print(
    "Normal after last defect:",
    int(
        (
            future_after_failure[
                "PassOrFail"
            ] == 0
        ).sum()
    )
)

In [ ]:
# ============================================================
# CN7 FULL DATA - Failure Episode 구조 탐색
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. 전체 CN7 로드
# ============================================================

# 이미 DataFrame으로 불러왔다면 이 줄만 바꾸면 됨.
# 예: cn7_full = supervised_label_cn7.copy()

cn7_full = pd.read_csv(
    r"../data/raw/supervised_label_cn7.csv"
)

cn7_full["TimeStamp"] = pd.to_datetime(
    cn7_full["TimeStamp"]
)

cn7_full = (
    cn7_full
    .sort_values(
        ["TimeStamp", "Side"]
    )
    .reset_index(drop=True)
)


print("=" * 100)
print("CN7 FULL DATA CHECK")
print("=" * 100)

print("Shape :", cn7_full.shape)

print(
    "\nTarget:"
)

print(
    cn7_full[
        "PassOrFail"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nTime Range:"
)

print(
    cn7_full["TimeStamp"].min(),
    "~",
    cn7_full["TimeStamp"].max()
)

print(
    "\nUnique Timestamp:",
    cn7_full[
        "TimeStamp"
    ].nunique()
)


# ============================================================
# 1. 날짜별 생산 / 불량 현황
# ============================================================

cn7_full[
    "Date"
] = (
    cn7_full[
        "TimeStamp"
    ]
    .dt.date
)


date_summary = (
    cn7_full
    .groupby(
        "Date"
    )
    .agg(
        Rows=(
            "PassOrFail",
            "size"
        ),

        Defect_Rows=(
            "PassOrFail",
            "sum"
        ),

        Normal_Rows=(
            "PassOrFail",
            lambda x:
            (x == 0).sum()
        ),

        Unique_Cycles=(
            "TimeStamp",
            "nunique"
        ),

        Start_Time=(
            "TimeStamp",
            "min"
        ),

        End_Time=(
            "TimeStamp",
            "max"
        )
    )
    .reset_index()
)


print(
    "\n"
    + "=" * 100
)

print(
    "DATE SUMMARY"
)

print(
    "=" * 100
)

print(
    date_summary
    .to_string(
        index=False
    )
)


# ============================================================
# 2. TimeStamp = 생산 Cycle 단위로 압축
#
# 동일 Cycle에서 LH/RH 중 하나라도 불량이면
# 해당 Cycle을 불량 Cycle로 정의
# ============================================================

cycle_df = (
    cn7_full
    .groupby(
        "TimeStamp",
        as_index=False
    )
    .agg(
        Cycle_Defect=(
            "PassOrFail",
            "max"
        ),

        Defect_Rows=(
            "PassOrFail",
            "sum"
        ),

        Row_Count=(
            "PassOrFail",
            "size"
        )
    )
    .sort_values(
        "TimeStamp"
    )
    .reset_index(drop=True)
)


cycle_df[
    "Cycle_Index"
] = np.arange(
    len(cycle_df)
)


cycle_df[
    "Date"
] = (
    cycle_df[
        "TimeStamp"
    ]
    .dt.date
)


print(
    "\n"
    + "=" * 100
)

print(
    "CYCLE SUMMARY"
)

print(
    "=" * 100
)

print(
    "전체 Cycle:",
    len(cycle_df)
)

print(
    "불량 Cycle:",
    int(
        cycle_df[
            "Cycle_Defect"
        ].sum()
    )
)

print(
    "불량 Row:",
    int(
        cn7_full[
            "PassOrFail"
        ].sum()
    )
)


# ============================================================
# 3. 불량 Cycle 추출
# ============================================================

defect_cycles = (
    cycle_df[
        cycle_df[
            "Cycle_Defect"
        ] == 1
    ]
    .copy()
    .reset_index(drop=True)
)


defect_cycles[
    "Gap_From_Previous_Defect_Min"
] = (
    defect_cycles[
        "TimeStamp"
    ]
    .diff()
    .dt.total_seconds()
    / 60
)


defect_cycles[
    "Cycle_Gap"
] = (
    defect_cycles[
        "Cycle_Index"
    ]
    .diff()
)


# 두 불량 사이에 몇 개의 정상 Cycle이 있었는지
defect_cycles[
    "Normal_Cycles_Between"
] = (
    defect_cycles[
        "Cycle_Gap"
    ]
    - 1
)


print(
    "\n"
    + "=" * 100
)

print(
    "FULL CN7 DEFECT CYCLE TIMELINE"
)

print(
    "=" * 100
)

print(
    defect_cycles[
        [
            "Cycle_Index",
            "TimeStamp",
            "Defect_Rows",
            "Gap_From_Previous_Defect_Min",
            "Cycle_Gap",
            "Normal_Cycles_Between"
        ]
    ]
    .round(
        {
            "Gap_From_Previous_Defect_Min": 2
        }
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 4. 날짜별 불량 Cycle 수
# ============================================================

defect_date_summary = (
    defect_cycles
    .groupby(
        "Date"
    )
    .agg(
        Defect_Cycles=(
            "Cycle_Defect",
            "size"
        ),

        Defect_Rows=(
            "Defect_Rows",
            "sum"
        ),

        First_Defect=(
            "TimeStamp",
            "min"
        ),

        Last_Defect=(
            "TimeStamp",
            "max"
        )
    )
    .reset_index()
)


print(
    "\n"
    + "=" * 100
)

print(
    "DEFECT DATE SUMMARY"
)

print(
    "=" * 100
)

print(
    defect_date_summary
    .to_string(
        index=False
    )
)


# ============================================================
# 5. Episode 생성 함수
#
# 이전 불량 Cycle과 일정 시간 이상 떨어져 있으면
# 새로운 Failure Episode로 간주
# ============================================================

def assign_failure_episode(
    defect_df,
    gap_threshold_min
):

    temp = (
        defect_df
        .copy()
    )

    new_episode = (
        temp[
            "Gap_From_Previous_Defect_Min"
        ]
        .isna()
        |
        (
            temp[
                "Gap_From_Previous_Defect_Min"
            ]
            > gap_threshold_min
        )
    )

    temp[
        "Episode_ID"
    ] = (
        new_episode
        .cumsum()
    )

    return temp


# ============================================================
# 6. 여러 Episode Gap 기준 비교
# ============================================================

candidate_thresholds = [
    2,
    5,
    10,
    30,
    60
]


threshold_results = []


for threshold in candidate_thresholds:

    temp = assign_failure_episode(
        defect_cycles,
        threshold
    )

    summary = (
        temp
        .groupby(
            "Episode_ID"
        )
        .agg(
            Start_Time=(
                "TimeStamp",
                "min"
            ),

            End_Time=(
                "TimeStamp",
                "max"
            ),

            Defect_Cycles=(
                "Cycle_Defect",
                "size"
            ),

            Defect_Rows=(
                "Defect_Rows",
                "sum"
            ),

            Start_Cycle=(
                "Cycle_Index",
                "min"
            ),

            End_Cycle=(
                "Cycle_Index",
                "max"
            )
        )
        .reset_index()
    )


    summary[
        "Duration_Min"
    ] = (
        summary[
            "End_Time"
        ]
        - summary[
            "Start_Time"
        ]
    ).dt.total_seconds() / 60


    threshold_results.append(
        {
            "Gap_Threshold_Min":
                threshold,

            "Episode_Count":
                len(summary),

            "Max_Defect_Cycles":
                summary[
                    "Defect_Cycles"
                ].max(),

            "Min_Defect_Cycles":
                summary[
                    "Defect_Cycles"
                ].min(),

            "Mean_Defect_Cycles":
                summary[
                    "Defect_Cycles"
                ].mean()
        }
    )


threshold_summary = pd.DataFrame(
    threshold_results
)


print(
    "\n"
    + "=" * 100
)

print(
    "FAILURE EPISODE THRESHOLD COMPARISON"
)

print(
    "=" * 100
)

print(
    threshold_summary
    .round(2)
    .to_string(
        index=False
    )
)


# ============================================================
# 7. 각 Threshold별 실제 Episode 상세 출력
# ============================================================

for threshold in candidate_thresholds:

    temp = assign_failure_episode(
        defect_cycles,
        threshold
    )

    detail = (
        temp
        .groupby(
            "Episode_ID"
        )
        .agg(
            Start_Time=(
                "TimeStamp",
                "min"
            ),

            End_Time=(
                "TimeStamp",
                "max"
            ),

            Defect_Cycles=(
                "Cycle_Defect",
                "size"
            ),

            Defect_Rows=(
                "Defect_Rows",
                "sum"
            ),

            Start_Cycle=(
                "Cycle_Index",
                "min"
            ),

            End_Cycle=(
                "Cycle_Index",
                "max"
            )
        )
        .reset_index()
    )


    detail[
        "Duration_Min"
    ] = (
        detail[
            "End_Time"
        ]
        - detail[
            "Start_Time"
        ]
    ).dt.total_seconds() / 60


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"EPISODE DETAIL | GAP > {threshold} MIN"
    )

    print(
        "=" * 100
    )

    print(
        detail
        .round(
            {
                "Duration_Min": 2
            }
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 8. 불량 발생 Timeline 시각화
# ============================================================

plt.figure(
    figsize=(15, 5)
)


plt.scatter(
    cycle_df[
        "TimeStamp"
    ],
    cycle_df[
        "Cycle_Defect"
    ],
    s=10,
    alpha=0.5
)


plt.scatter(
    defect_cycles[
        "TimeStamp"
    ],
    np.ones(
        len(defect_cycles)
    ),
    s=90
)


plt.yticks(
    [0, 1],
    [
        "Normal",
        "Defect"
    ]
)


plt.xlabel(
    "Production Time"
)

plt.ylabel(
    "Cycle Status"
)

plt.title(
    "CN7 Full Dataset - Defect Occurrence Timeline"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# 9. 불량 Cycle 간 시간 Gap
# ============================================================

plt.figure(
    figsize=(15, 5)
)


plt.plot(
    defect_cycles[
        "TimeStamp"
    ],
    defect_cycles[
        "Gap_From_Previous_Defect_Min"
    ],
    marker="o"
)


plt.axhline(
    y=5,
    linestyle="--",
    label="5 min"
)


plt.axhline(
    y=10,
    linestyle="--",
    label="10 min"
)


plt.xlabel(
    "Defect Time"
)

plt.ylabel(
    "Minutes From Previous Defect"
)

plt.title(
    "CN7 Full Dataset - Time Gap Between Defect Cycles"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
print(cn7_full.shape)
print(cn7_full.columns.tolist())

In [ ]:
# ============================================================
# CN7 FULL DATA - 컬럼명 확인 및 자동 정리
# ============================================================

import pandas as pd

print("Shape:", cn7_full.shape)

print("\n전체 컬럼")
for i, col in enumerate(cn7_full.columns):
    print(i, repr(col))


# ------------------------------------------------------------
# 1. 컬럼명 앞뒤 공백 제거
# ------------------------------------------------------------

cn7_full.columns = (
    cn7_full.columns
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# 2. 시간 컬럼 후보 자동 탐색
# ------------------------------------------------------------

time_candidates = [
    col
    for col in cn7_full.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "timestamp",
            "time_stamp",
            "datetime",
            "date_time",
            "date",
            "time"
        ]
    )
]


print("\n시간 컬럼 후보:")
print(time_candidates)


# ------------------------------------------------------------
# 3. Target 후보 확인
# ------------------------------------------------------------

target_candidates = [
    col
    for col in cn7_full.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "passorfail",
            "pass_or_fail",
            "target",
            "label",
            "fail"
        ]
    )
]


print("\nTarget 컬럼 후보:")
print(target_candidates)


# ------------------------------------------------------------
# 4. Side 후보 확인
# ------------------------------------------------------------

side_candidates = [
    col
    for col in cn7_full.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "side",
            "rh_lh",
            "lh_rh"
        ]
    )
]


print("\nSide 컬럼 후보:")
print(side_candidates)

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1. 원본 labeled_data 로드
# ============================================================

raw = pd.read_csv(r"../../data/raw/labeled_data.csv")

raw["TimeStamp"] = pd.to_datetime(
    raw["TimeStamp"]
)

print("원본 shape:", raw.shape)


# ============================================================
# 2. CN7만 추출
# ============================================================

cn7_full = (
    raw[
        raw["PART_NAME"]
        .astype(str)
        .str.contains("CN7", na=False)
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 3. Side 복원
# ============================================================

cn7_full["Side"] = np.select(
    [
        cn7_full["PART_NAME"]
        .astype(str)
        .str.contains("RH", na=False),

        cn7_full["PART_NAME"]
        .astype(str)
        .str.contains("LH", na=False)
    ],
    [
        "RH",
        "LH"
    ],
    default="UNKNOWN"
)


cn7_full["Side_RH"] = (
    cn7_full["Side"] == "RH"
).astype(int)


# ============================================================
# 4. 시간순 정렬
# ============================================================

cn7_full = (
    cn7_full
    .sort_values(
        [
            "TimeStamp",
            "Side"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 5. 복원 확인
# ============================================================

print("\n" + "=" * 100)
print("CN7 FULL RESTORED")
print("=" * 100)

print("Shape:", cn7_full.shape)

print(
    "\n기간:",
    cn7_full["TimeStamp"].min(),
    "~",
    cn7_full["TimeStamp"].max()
)

print("\nSide:")
print(
    cn7_full["Side"]
    .value_counts()
)

print("\nTarget:")
print(
    cn7_full["PassOrFail"]
    .value_counts()
    .sort_index()
)

print(
    "\nUnique TimeStamp:",
    cn7_full["TimeStamp"].nunique()
)

print(
    "\nDefect Rows:",
    int(cn7_full["PassOrFail"].sum())
)

print(
    "\n불량 날짜:"
)

print(
    cn7_full.loc[
        cn7_full["PassOrFail"] == 1,
        "TimeStamp"
    ]
    .dt.date
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# CN7 FULL - PassOrFail Y/N -> 0/1 변환
# ============================================================

# 원본 보존
cn7_full["PassOrFail_raw"] = (
    cn7_full["PassOrFail"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("=" * 80)
print("원본 PassOrFail 분포")
print("=" * 80)

print(
    cn7_full["PassOrFail_raw"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# Y = 정상(0)
# N = 불량(1)
# ------------------------------------------------------------

target_map = {
    "Y": 0,
    "N": 1
}

cn7_full["PassOrFail"] = (
    cn7_full["PassOrFail_raw"]
    .map(target_map)
)


# ------------------------------------------------------------
# 매핑 실패 확인
# ------------------------------------------------------------

print("\n매핑 실패 개수:")
print(
    cn7_full["PassOrFail"]
    .isna()
    .sum()
)

print("\n변환 후 Target 분포:")
print(
    cn7_full["PassOrFail"]
    .value_counts()
    .sort_index()
)


print(
    "\n불량 Row 수:",
    int(
        cn7_full["PassOrFail"]
        .sum()
    )
)

In [ ]:
# ============================================================
# CN7 FULL - Cycle 단위 불량 구조 확인
# ============================================================

cycle_df = (
    cn7_full
    .groupby(
        "TimeStamp",
        as_index=False
    )
    .agg(
        Cycle_Defect=(
            "PassOrFail",
            "max"
        ),

        Defect_Rows=(
            "PassOrFail",
            "sum"
        ),

        Row_Count=(
            "PassOrFail",
            "size"
        )
    )
    .sort_values(
        "TimeStamp"
    )
    .reset_index(drop=True)
)


cycle_df["Cycle_Index"] = np.arange(
    len(cycle_df)
)


# 불량 Cycle만 추출
defect_cycles = (
    cycle_df[
        cycle_df["Cycle_Defect"] == 1
    ]
    .copy()
    .reset_index(drop=True)
)


# 이전 불량 Cycle과 시간 차이
defect_cycles[
    "Gap_From_Previous_Defect_Min"
] = (
    defect_cycles["TimeStamp"]
    .diff()
    .dt.total_seconds()
    / 60
)


# Cycle 순서 차이
defect_cycles[
    "Cycle_Gap"
] = (
    defect_cycles["Cycle_Index"]
    .diff()
)


# 중간에 존재한 정상 Cycle 수
defect_cycles[
    "Normal_Cycles_Between"
] = (
    defect_cycles["Cycle_Gap"]
    - 1
)


print("\n" + "=" * 120)
print("CN7 FULL DEFECT CYCLE TIMELINE")
print("=" * 120)

print(
    defect_cycles[
        [
            "Cycle_Index",
            "TimeStamp",
            "Defect_Rows",
            "Gap_From_Previous_Defect_Min",
            "Cycle_Gap",
            "Normal_Cycles_Between"
        ]
    ]
    .round(
        {
            "Gap_From_Previous_Defect_Min": 2
        }
    )
    .to_string(
        index=False
    )
)


print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

print(
    "전체 Rows:",
    len(cn7_full)
)

print(
    "전체 Cycle:",
    len(cycle_df)
)

print(
    "불량 Rows:",
    int(
        cn7_full["PassOrFail"]
        .sum()
    )
)

print(
    "불량 Cycles:",
    len(defect_cycles)
)

In [ ]:
# ============================================================
# 날짜별 불량 구조
# ============================================================

defect_date_summary = (
    cn7_full[
        cn7_full["PassOrFail"] == 1
    ]
    .assign(
        Date=lambda x:
        x["TimeStamp"].dt.date
    )
    .groupby("Date")
    .agg(
        Defect_Rows=(
            "PassOrFail",
            "size"
        ),

        Defect_Cycles=(
            "TimeStamp",
            "nunique"
        ),

        First_Defect=(
            "TimeStamp",
            "min"
        ),

        Last_Defect=(
            "TimeStamp",
            "max"
        )
    )
    .reset_index()
)


print("\n" + "=" * 100)
print("CN7 DEFECT DATE SUMMARY")
print("=" * 100)

print(
    defect_date_summary
    .to_string(index=False)
)

In [ ]:
# ============================================================
# CN7 FULL - Timestamp / Side 중복 구조 확인
# ============================================================

# TimeStamp당 행 개수 분포
timestamp_counts = (
    cn7_full
    .groupby("TimeStamp")
    .size()
)

print("=" * 100)
print("ROWS PER TIMESTAMP")
print("=" * 100)

print(
    timestamp_counts
    .value_counts()
    .sort_index()
)


# TimeStamp + Side 당 행 개수
timestamp_side_counts = (
    cn7_full
    .groupby(
        [
            "TimeStamp",
            "Side"
        ]
    )
    .size()
    .reset_index(
        name="Row_Count"
    )
)


print("\n" + "=" * 100)
print("ROWS PER TIMESTAMP + SIDE")
print("=" * 100)

print(
    timestamp_side_counts[
        "Row_Count"
    ]
    .value_counts()
    .sort_index()
)


# 2개 이상 존재하는 TimeStamp + Side
duplicated_ts_side = (
    timestamp_side_counts[
        timestamp_side_counts[
            "Row_Count"
        ] > 1
    ]
    .copy()
)


print(
    "\nTimeStamp + Side 중복 조합 수:",
    len(duplicated_ts_side)
)


print("\n" + "=" * 100)
print("DEFECT TIMESTAMP DETAIL")
print("=" * 100)


defect_times = (
    defect_cycles[
        "TimeStamp"
    ]
    .tolist()
)


detail_cols = [
    col
    for col in [
        "TimeStamp",
        "PART_NAME",
        "Side",
        "PassOrFail"
    ]
    if col in cn7_full.columns
]


print(
    cn7_full[
        cn7_full[
            "TimeStamp"
        ].isin(
            defect_times
        )
    ][
        detail_cols
    ]
    .sort_values(
        [
            "TimeStamp",
            "Side"
        ]
    )
    .to_string(
        index=False
    )
)

In [ ]:
# ============================================================
# CN7 FULL - 5분 기준 Failure Episode 부여
# ============================================================

EPISODE_GAP_MIN = 5


defect_episode_df = (
    defect_cycles
    .copy()
)


# 첫 불량이거나
# 이전 불량과 5분 초과 떨어졌으면 새로운 Episode
new_episode = (
    defect_episode_df[
        "Gap_From_Previous_Defect_Min"
    ]
    .isna()
    |
    (
        defect_episode_df[
            "Gap_From_Previous_Defect_Min"
        ]
        > EPISODE_GAP_MIN
    )
)


defect_episode_df[
    "Episode_ID"
] = (
    new_episode
    .cumsum()
)


defect_episode_df[
    "Episode_ID"
] = (
    "E"
    +
    defect_episode_df[
        "Episode_ID"
    ]
    .astype(str)
)


# ============================================================
# Episode Summary
# ============================================================

episode_summary = (
    defect_episode_df
    .groupby(
        "Episode_ID"
    )
    .agg(
        Start_Time=(
            "TimeStamp",
            "min"
        ),

        End_Time=(
            "TimeStamp",
            "max"
        ),

        Defect_Cycles=(
            "Cycle_Defect",
            "size"
        ),

        Defect_Rows=(
            "Defect_Rows",
            "sum"
        ),

        Start_Cycle=(
            "Cycle_Index",
            "min"
        ),

        End_Cycle=(
            "Cycle_Index",
            "max"
        )
    )
    .reset_index()
)


episode_summary[
    "Duration_Min"
] = (
    episode_summary[
        "End_Time"
    ]
    -
    episode_summary[
        "Start_Time"
    ]
).dt.total_seconds() / 60


print(
    "\n"
    + "=" * 110
)

print(
    "CN7 FAILURE EPISODE SUMMARY"
)

print(
    "=" * 110
)

print(
    episode_summary
    .round(
        {
            "Duration_Min": 2
        }
    )
    .to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "CN7 FAILURE EPISODE DETAIL"
)

print(
    "=" * 110
)

print(
    defect_episode_df[
        [
            "Episode_ID",
            "Cycle_Index",
            "TimeStamp",
            "Defect_Rows",
            "Gap_From_Previous_Defect_Min",
            "Normal_Cycles_Between"
        ]
    ]
    .round(
        {
            "Gap_From_Previous_Defect_Min": 2
        }
    )
    .to_string(
        index=False
    )
)

In [ ]:
# ============================================================
# P1.8 - CN7 Failure Episode Sensor Pattern Comparison
#
# 목적
# 1. 각 positive cycle에 Episode_ID 부여
# 2. Episode별 센서 평균/중앙값 확인
# 3. E3 vs E4 패턴 차이 정량화
# 4. 각 Episode가 정상 생산분포에서 얼마나 벗어나는지 확인
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 0. 사용할 센서 Feature
# ============================================================

SENSOR_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4"
]

SENSOR_COLS = [
    c for c in SENSOR_COLS
    if c in cn7_full.columns
]

print("Sensor Feature 수:", len(SENSOR_COLS))


# ============================================================
# 1. Episode_ID를 TimeStamp 기준으로 원본 row에 매핑
# ============================================================

episode_map = (
    defect_episode_df[
        [
            "TimeStamp",
            "Episode_ID"
        ]
    ]
    .drop_duplicates()
)


cn7_episode = (
    cn7_full
    .merge(
        episode_map,
        on="TimeStamp",
        how="left"
    )
)


# positive row만 Episode_ID 유지
cn7_episode.loc[
    cn7_episode["PassOrFail"] == 0,
    "Episode_ID"
] = np.nan


print("\nEpisode별 불량 row 수")

print(
    cn7_episode[
        cn7_episode["PassOrFail"] == 1
    ]
    ["Episode_ID"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 2. 정상 reference
# ============================================================

normal_df = (
    cn7_episode[
        cn7_episode["PassOrFail"] == 0
    ][SENSOR_COLS]
    .copy()
)


normal_mean = (
    normal_df
    .mean()
)

normal_std = (
    normal_df
    .std()
    .replace(0, np.nan)
)

normal_median = (
    normal_df
    .median()
)


# ============================================================
# 3. Episode별 센서 평균
# ============================================================

episode_feature_mean = (
    cn7_episode[
        cn7_episode["PassOrFail"] == 1
    ]
    .groupby("Episode_ID")[SENSOR_COLS]
    .mean()
)


episode_feature_median = (
    cn7_episode[
        cn7_episode["PassOrFail"] == 1
    ]
    .groupby("Episode_ID")[SENSOR_COLS]
    .median()
)


print("\n" + "=" * 110)
print("EPISODE FEATURE MEAN")
print("=" * 110)

print(
    episode_feature_mean
    .round(4)
    .to_string()
)


# ============================================================
# 4. 정상 대비 Episode Z-distance
#
# 정상 평균에서 Episode 평균이
# 몇 표준편차 떨어져 있는지
# ============================================================

episode_z = (
    episode_feature_mean
    .subtract(
        normal_mean,
        axis=1
    )
    .divide(
        normal_std,
        axis=1
    )
)


episode_abs_z = (
    episode_z
    .abs()
)


print("\n" + "=" * 110)
print("EPISODE vs NORMAL | TOP DEVIATING FEATURES")
print("=" * 110)


for episode in episode_abs_z.index:

    temp = (
        pd.DataFrame({
            "Feature":
                SENSOR_COLS,

            "Episode_Mean":
                episode_feature_mean.loc[
                    episode,
                    SENSOR_COLS
                ].values,

            "Normal_Mean":
                normal_mean[
                    SENSOR_COLS
                ].values,

            "Z_Diff":
                episode_z.loc[
                    episode,
                    SENSOR_COLS
                ].values,

            "Abs_Z":
                episode_abs_z.loc[
                    episode,
                    SENSOR_COLS
                ].values
        })
        .sort_values(
            "Abs_Z",
            ascending=False
        )
    )

    print(
        f"\n[{episode}]"
    )

    print(
        temp
        .head(10)
        .round(4)
        .to_string(index=False)
    )


# ============================================================
# 5. Episode 간 패턴 유사도
#
# Episode별 정상 대비 Z 패턴을 이용
# Pearson correlation 계산
# ============================================================

episode_pattern_corr = (
    episode_z
    .T
    .corr()
)


print("\n" + "=" * 110)
print("EPISODE PATTERN CORRELATION")
print("=" * 110)

print(
    episode_pattern_corr
    .round(4)
    .to_string()
)


# ============================================================
# 6. Episode 간 Euclidean Distance
#
# Z-score pattern 기준 거리
# ============================================================

episodes = (
    episode_z
    .index
    .tolist()
)


distance_rows = []


for i in range(len(episodes)):

    for j in range(
        i + 1,
        len(episodes)
    ):

        e1 = episodes[i]
        e2 = episodes[j]

        v1 = (
            episode_z
            .loc[e1]
            .fillna(0)
            .values
        )

        v2 = (
            episode_z
            .loc[e2]
            .fillna(0)
            .values
        )

        distance = (
            np.linalg.norm(
                v1 - v2
            )
        )

        correlation = (
            np.corrcoef(
                v1,
                v2
            )[0, 1]
        )

        distance_rows.append({
            "Episode_1":
                e1,

            "Episode_2":
                e2,

            "Euclidean_Distance":
                distance,

            "Pattern_Correlation":
                correlation
        })


episode_distance_df = (
    pd.DataFrame(
        distance_rows
    )
    .sort_values(
        "Euclidean_Distance"
    )
)


print("\n" + "=" * 110)
print("EPISODE PAIR DISTANCE")
print("=" * 110)

print(
    episode_distance_df
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 7. E3 vs E4 상세 비교
# ============================================================

if (
    "E3" in episode_feature_mean.index
    and
    "E4" in episode_feature_mean.index
):

    e3_e4 = (
        pd.DataFrame({
            "Feature":
                SENSOR_COLS,

            "E3_Mean":
                episode_feature_mean
                .loc[
                    "E3",
                    SENSOR_COLS
                ]
                .values,

            "E4_Mean":
                episode_feature_mean
                .loc[
                    "E4",
                    SENSOR_COLS
                ]
                .values,

            "E3_Z":
                episode_z
                .loc[
                    "E3",
                    SENSOR_COLS
                ]
                .values,

            "E4_Z":
                episode_z
                .loc[
                    "E4",
                    SENSOR_COLS
                ]
                .values
        })
    )


    e3_e4[
        "Z_Difference"
    ] = (
        e3_e4[
            "E4_Z"
        ]
        -
        e3_e4[
            "E3_Z"
        ]
    )


    e3_e4[
        "Abs_Z_Difference"
    ] = (
        e3_e4[
            "Z_Difference"
        ]
        .abs()
    )


    e3_e4 = (
        e3_e4
        .sort_values(
            "Abs_Z_Difference",
            ascending=False
        )
    )


    print("\n" + "=" * 110)
    print("E3 vs E4 | LARGEST SENSOR PATTERN DIFFERENCE")
    print("=" * 110)

    print(
        e3_e4
        .head(15)
        .round(4)
        .to_string(index=False)
    )


# ============================================================
# 8. Episode 특성 요약
# ============================================================

episode_characteristics = []


for episode in episode_z.index:

    z = (
        episode_z
        .loc[episode]
        .dropna()
    )

    episode_characteristics.append({
        "Episode_ID":
            episode,

        "Mean_Abs_Z":
            z.abs().mean(),

        "Max_Abs_Z":
            z.abs().max(),

        "Features_AbsZ_gt_1":
            int(
                (z.abs() > 1).sum()
            ),

        "Features_AbsZ_gt_2":
            int(
                (z.abs() > 2).sum()
            ),

        "Features_AbsZ_gt_3":
            int(
                (z.abs() > 3).sum()
            )
    })


episode_characteristics = (
    pd.DataFrame(
        episode_characteristics
    )
)


print("\n" + "=" * 110)
print("EPISODE ABNORMALITY SUMMARY")
print("=" * 110)

print(
    episode_characteristics
    .round(4)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# P1.8-1
# CN7 Full - TimeStamp / Side 중복 구조 확인
# ============================================================

# 1) TimeStamp 당 row 수
ts_count = (
    cn7_full
    .groupby("TimeStamp")
    .size()
)

print("=" * 100)
print("ROWS PER TIMESTAMP")
print("=" * 100)
print(ts_count.value_counts().sort_index())


# 2) TimeStamp + Side 당 row 수
ts_side_count = (
    cn7_full
    .groupby(["TimeStamp", "Side"])
    .size()
    .reset_index(name="Row_Count")
)

print("\n" + "=" * 100)
print("ROWS PER TIMESTAMP + SIDE")
print("=" * 100)

print(
    ts_side_count["Row_Count"]
    .value_counts()
    .sort_index()
)


# 3) 동일 TimeStamp + Side에 여러 row 존재하는 조합
dup_ts_side = (
    ts_side_count[
        ts_side_count["Row_Count"] > 1
    ]
    .copy()
)

print(
    "\n중복 TimeStamp + Side 조합:",
    len(dup_ts_side)
)


# 4) 불량 timestamp의 실제 row 상세
defect_times = (
    defect_episode_df["TimeStamp"]
    .unique()
)

show_cols = [
    c for c in [
        "TimeStamp",
        "PART_NAME",
        "Side",
        "PassOrFail",
        "Injection_Time",
        "Filling_Time",
        "Max_Injection_Speed",
        "Max_Injection_Pressure",
        "Mold_Temperature_3",
        "Mold_Temperature_4"
    ]
    if c in cn7_full.columns
]

defect_detail = (
    cn7_full[
        cn7_full["TimeStamp"].isin(defect_times)
    ][show_cols]
    .sort_values(["TimeStamp", "Side"])
)

print("\n" + "=" * 100)
print("DEFECT TIMESTAMP ROW DETAIL")
print("=" * 100)

print(
    defect_detail.to_string(index=False)
)


# 5) 동일 TimeStamp + Side 내부 센서값이 완전 중복인지 검사
sensor_cols_check = SENSOR_COLS.copy()

dup_check = (
    cn7_full
    .groupby(["TimeStamp", "Side"])
    .agg(
        Rows=("PassOrFail", "size"),
        Unique_Target=("PassOrFail", "nunique")
    )
    .reset_index()
)

# 센서 전체 조합 중복 여부를 별도로 계산
sensor_unique = (
    cn7_full[
        ["TimeStamp", "Side"] + sensor_cols_check
    ]
    .drop_duplicates()
    .groupby(["TimeStamp", "Side"])
    .size()
    .reset_index(name="Unique_Sensor_Rows")
)

dup_check = (
    dup_check
    .merge(
        sensor_unique,
        on=["TimeStamp", "Side"],
        how="left"
    )
)

print("\n" + "=" * 100)
print("DUPLICATE STRUCTURE SUMMARY")
print("=" * 100)

print(
    dup_check[
        dup_check["Rows"] > 1
    ][
        [
            "TimeStamp",
            "Side",
            "Rows",
            "Unique_Sensor_Rows",
            "Unique_Target"
        ]
    ]
    .head(50)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# P1.8-2
# CN7 중복 레코드 최종 검증
# + TimeStamp × Side 단위 데이터 생성
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. 모델링에 사용할 센서 컬럼
# ============================================================

SENSOR_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4"
]

SENSOR_COLS = [
    c for c in SENSOR_COLS
    if c in cn7_full.columns
]


# ============================================================
# 2. TimeStamp + Side 그룹별 구조 확인
# ============================================================

group_rows = []

for (ts, side), g in cn7_full.groupby(
    ["TimeStamp", "Side"]
):

    # 센서 조합 종류 수
    sensor_unique = (
        g[SENSOR_COLS]
        .drop_duplicates()
        .shape[0]
    )

    # Target 종류 수
    target_unique = (
        g["PassOrFail"]
        .nunique()
    )

    group_rows.append({
        "TimeStamp": ts,
        "Side": side,
        "Rows": len(g),
        "Unique_Sensor_Rows": sensor_unique,
        "Unique_Target": target_unique
    })


group_check = pd.DataFrame(
    group_rows
)


print("=" * 100)
print("TIMESTAMP + SIDE STRUCTURE")
print("=" * 100)

print(
    group_check[
        [
            "Rows",
            "Unique_Sensor_Rows",
            "Unique_Target"
        ]
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 3. 문제가 있는 그룹 탐색
#
# 같은 TimeStamp + Side인데
# 센서가 다르거나 Target이 다른 경우
# ============================================================

problem_groups = (
    group_check[
        (group_check["Unique_Sensor_Rows"] > 1)
        |
        (group_check["Unique_Target"] > 1)
    ]
    .copy()
)


print("\n" + "=" * 100)
print("NON-IDENTICAL DUPLICATE GROUPS")
print("=" * 100)

print(
    "문제 그룹 수:",
    len(problem_groups)
)


if len(problem_groups) > 0:

    print(
        problem_groups
        .head(30)
        .to_string(index=False)
    )

else:

    print(
        "모든 TimeStamp + Side 중복은 "
        "센서 및 Target 기준 동일함."
    )


# ============================================================
# 4. 중복 제거
#
# 센서와 Target이 동일하다는 전제에서
# TimeStamp + Side당 한 행 유지
# ============================================================

if len(problem_groups) == 0:

    cn7_unit = (
        cn7_full
        .sort_values(
            ["TimeStamp", "Side"]
        )
        .drop_duplicates(
            subset=[
                "TimeStamp",
                "Side"
            ],
            keep="first"
        )
        .reset_index(drop=True)
    )

else:

    print(
        "\n주의: 서로 다른 센서/Target을 가진 "
        "TimeStamp + Side가 존재하므로 자동 제거 중단."
    )

    cn7_unit = None


# ============================================================
# 5. 중복 제거 전후 비교
# ============================================================

if cn7_unit is not None:

    print("\n" + "=" * 100)
    print("BEFORE / AFTER DEDUPLICATION")
    print("=" * 100)

    print(
        f"Raw rows               : {len(cn7_full)}"
    )

    print(
        f"Unique TimeStamp+Side   : {len(cn7_unit)}"
    )

    print(
        f"Removed duplicate rows  : "
        f"{len(cn7_full) - len(cn7_unit)}"
    )

    print(
        f"Removal rate            : "
        f"{(1 - len(cn7_unit)/len(cn7_full))*100:.2f}%"
    )


    print("\nTarget BEFORE")

    print(
        cn7_full[
            "PassOrFail"
        ]
        .value_counts()
        .sort_index()
    )


    print("\nTarget AFTER")

    print(
        cn7_unit[
            "PassOrFail"
        ]
        .value_counts()
        .sort_index()
    )


# ============================================================
# 6. Cycle-level 구조 재계산
# ============================================================

if cn7_unit is not None:

    cycle_unit = (
        cn7_unit
        .groupby(
            "TimeStamp",
            as_index=False
        )
        .agg(
            Cycle_Defect=(
                "PassOrFail",
                "max"
            ),

            Defect_Sides=(
                "PassOrFail",
                "sum"
            ),

            Side_Records=(
                "PassOrFail",
                "size"
            )
        )
        .sort_values(
            "TimeStamp"
        )
        .reset_index(drop=True)
    )


    cycle_unit[
        "Cycle_Index"
    ] = np.arange(
        len(cycle_unit)
    )


    positive_cycles_unit = (
        cycle_unit[
            cycle_unit[
                "Cycle_Defect"
            ] == 1
        ]
        .copy()
    )


    print("\n" + "=" * 100)
    print("DEDUPLICATED CYCLE SUMMARY")
    print("=" * 100)

    print(
        "전체 Cycle:",
        len(cycle_unit)
    )

    print(
        "Positive Cycle:",
        len(positive_cycles_unit)
    )

    print(
        "Defect Side Records:",
        int(
            cn7_unit[
                "PassOrFail"
            ].sum()
        )
    )


# ============================================================
# 7. Episode별 실제 Side-level 불량 수 재계산
# ============================================================

if cn7_unit is not None:

    episode_time_map = (
        defect_episode_df[
            [
                "TimeStamp",
                "Episode_ID"
            ]
        ]
        .drop_duplicates()
    )


    defect_unit_episode = (
        cn7_unit[
            cn7_unit[
                "PassOrFail"
            ] == 1
        ]
        .merge(
            episode_time_map,
            on="TimeStamp",
            how="left"
        )
    )


    episode_unit_summary = (
        defect_unit_episode
        .groupby(
            "Episode_ID"
        )
        .agg(
            Defect_Side_Records=(
                "PassOrFail",
                "size"
            ),

            Positive_Cycles=(
                "TimeStamp",
                "nunique"
            ),

            Start_Time=(
                "TimeStamp",
                "min"
            ),

            End_Time=(
                "TimeStamp",
                "max"
            )
        )
        .reset_index()
    )


    print("\n" + "=" * 100)
    print("DEDUPLICATED EPISODE SUMMARY")
    print("=" * 100)

    print(
        episode_unit_summary
        .to_string(index=False)
    )


# ============================================================
# 8. Cycle별 Side 구성 확인
# ============================================================

if cn7_unit is not None:

    side_coverage = (
        cn7_unit
        .groupby(
            "TimeStamp"
        )["Side"]
        .nunique()
    )


    print("\n" + "=" * 100)
    print("SIDE COVERAGE PER CYCLE")
    print("=" * 100)

    print(
        side_coverage
        .value_counts()
        .sort_index()
    )

In [ ]:
# ============================================================
# P1.9
# CN7 Episode / Time-Aware Temporal Validation
#
# 목적
# 1. deduplicated TimeStamp × Side 데이터 사용
# 2. 기존 47-feature recipe 유지
# 3. 과거 데이터로만 모델 학습
# 4. 미래 날짜/Failure Episode 평가
# 5. Risk Only vs Risk + Ensemble Disagreement 비교
# ============================================================

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ============================================================
# 0. 설정
# ============================================================

SEEDS = [
    42,
    100,
    2026,
    777,
    1234
]

RISK_THRESHOLD = 0.060

BEST_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.10,
    "num_leaves": 15,
    "max_depth": 7,
    "min_child_samples": 10,
    "subsample": 1.0,
    "colsample_bytree": 0.70,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0
}


SENSOR_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4"
]


# ============================================================
# 1. Side_RH 생성
# ============================================================

model_base = (
    cn7_unit
    .copy()
)

model_base["Side_RH"] = (
    model_base["Side"] == "RH"
).astype(int)


model_base = (
    model_base
    .sort_values(
        [
            "TimeStamp",
            "Side"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 2. Cycle-level 센서 동일성 확인
#
# 기존 47-feature recipe는
# 같은 TimeStamp의 공정 센서가 cycle-level 공통값이라는
# 구조에 기반했으므로 확인
# ============================================================

cycle_sensor_unique = (
    model_base[
        ["TimeStamp"] + SENSOR_COLS
    ]
    .drop_duplicates()
    .groupby("TimeStamp")
    .size()
)


conflict_cycle_count = int(
    (cycle_sensor_unique > 1).sum()
)


print("=" * 100)
print("CYCLE SENSOR CONSISTENCY CHECK")
print("=" * 100)

print(
    "동일 TimeStamp에서 센서 조합이 2개 이상인 Cycle:",
    conflict_cycle_count
)


# ============================================================
# 센서가 Side마다 다르다면 평균을 사용하지 않고 중단
# ============================================================

if conflict_cycle_count > 0:

    raise ValueError(
        "같은 TimeStamp 내부에서 Side별 센서값이 다른 Cycle이 존재합니다. "
        "47-feature recipe를 적용하기 전에 해당 Cycle 구조를 확인해야 합니다."
    )


# ============================================================
# 3. Cycle-level 데이터 생성
# ============================================================

cycle_sensor = (
    model_base[
        ["TimeStamp"] + SENSOR_COLS
    ]
    .drop_duplicates(
        subset=["TimeStamp"]
    )
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


# ============================================================
# 4. 생산 Session 생성
#
# 5분 초과 공백이면 새로운 session
# ============================================================

cycle_sensor["Time_Gap_Sec"] = (
    cycle_sensor["TimeStamp"]
    .diff()
    .dt.total_seconds()
)


cycle_sensor["session_id"] = (
    (
        cycle_sensor["Time_Gap_Sec"].isna()
        |
        (cycle_sensor["Time_Gap_Sec"] > 300)
    )
    .cumsum()
)


# ============================================================
# 5. 직전 5 Cycle Rolling Mean 생성
#
# 중요:
# shift(1)을 먼저 적용
# → 현재 Cycle 정보는 rolling mean에 포함하지 않음
# ============================================================

roll_mean_cols = []


for col in SENSOR_COLS:

    new_col = (
        f"{col}_roll5_mean"
    )

    cycle_sensor[new_col] = (
        cycle_sensor
        .groupby("session_id")[col]
        .transform(
            lambda s:
            s
            .shift(1)
            .rolling(
                window=5,
                min_periods=1
            )
            .mean()
        )
    )

    roll_mean_cols.append(
        new_col
    )


# ============================================================
# 6. Side-level 데이터에 Rolling Feature 병합
# ============================================================

rolling_merge_cols = (
    ["TimeStamp"]
    + roll_mean_cols
)


model_df = (
    model_base
    .merge(
        cycle_sensor[
            rolling_merge_cols
        ],
        on="TimeStamp",
        how="left"
    )
)


# ============================================================
# 7. 기존 후보 모델과 동일한 47 Features
# ============================================================

FEATURE_COLS = (
    SENSOR_COLS
    + ["Side_RH"]
    + roll_mean_cols
)


print("\n" + "=" * 100)
print("TEMPORAL MODEL DATA")
print("=" * 100)

print(
    "Rows:",
    len(model_df)
)

print(
    "Features:",
    len(FEATURE_COLS)
)

print(
    "Defect Side Records:",
    int(
        model_df[
            "PassOrFail"
        ].sum()
    )
)


# ============================================================
# 8. Episode ID 부여
#
# 불량 timestamp에만 Episode_ID 존재
# ============================================================

episode_map = (
    defect_episode_df[
        [
            "TimeStamp",
            "Episode_ID"
        ]
    ]
    .drop_duplicates()
)


model_df = (
    model_df
    .merge(
        episode_map,
        on="TimeStamp",
        how="left"
    )
)


# ============================================================
# 9. Temporal Fold 정의
#
# E4:
# Train = 10/27 이전
# Test  = 10/27 생산
#
# E5:
# Train = 11/03 이전
# Test  = 11/03 생산
# ============================================================

temporal_folds = [
    {
        "Fold": "Future_E4_2020-10-27",
        "Train_End":
            pd.Timestamp(
                "2020-10-27 00:00:00"
            ),
        "Test_Start":
            pd.Timestamp(
                "2020-10-27 00:00:00"
            ),
        "Test_End":
            pd.Timestamp(
                "2020-10-28 00:00:00"
            ),
        "Target_Episode":
            "E4"
    },

    {
        "Fold": "Future_E5_2020-11-03",
        "Train_End":
            pd.Timestamp(
                "2020-11-03 00:00:00"
            ),
        "Test_Start":
            pd.Timestamp(
                "2020-11-03 00:00:00"
            ),
        "Test_End":
            pd.Timestamp(
                "2020-11-04 00:00:00"
            ),
        "Target_Episode":
            "E5"
    }
]


# ============================================================
# 10. 평가 함수
# ============================================================

def metric_dict(
    y_true,
    pred,
    name
):

    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            pred,
            labels=[0, 1]
        )
        .ravel()
    )

    return {
        "Policy":
            name,

        "Precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "TP":
            int(tp),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TN":
            int(tn),

        "Flag_Count":
            int(
                np.sum(pred)
            ),

        "Flag_Rate_pct":
            (
                np.mean(pred)
                * 100
            )
    }


# ============================================================
# 11. 결과 저장 객체
# ============================================================

all_policy_results = []
all_predictions = []
cycle_results = []


# ============================================================
# 12. Temporal Validation
# ============================================================

for fold_info in temporal_folds:

    fold_name = (
        fold_info[
            "Fold"
        ]
    )

    train_end = (
        fold_info[
            "Train_End"
        ]
    )

    test_start = (
        fold_info[
            "Test_Start"
        ]
    )

    test_end = (
        fold_info[
            "Test_End"
        ]
    )

    target_episode = (
        fold_info[
            "Target_Episode"
        ]
    )


    # --------------------------------------------------------
    # Train / Test 분리
    # --------------------------------------------------------

    train_df = (
        model_df[
            model_df[
                "TimeStamp"
            ] < train_end
        ]
        .copy()
    )


    test_df = (
        model_df[
            (
                model_df[
                    "TimeStamp"
                ] >= test_start
            )
            &
            (
                model_df[
                    "TimeStamp"
                ] < test_end
            )
        ]
        .copy()
    )


    print(
        "\n\n"
        + "=" * 110
    )

    print(
        fold_name
    )

    print(
        "=" * 110
    )


    print(
        "Train:",
        train_df[
            "TimeStamp"
        ].min(),
        "~",
        train_df[
            "TimeStamp"
        ].max()
    )

    print(
        "Train rows:",
        len(train_df)
    )

    print(
        "Train defects:",
        int(
            train_df[
                "PassOrFail"
            ].sum()
        )
    )


    print(
        "\nTest:",
        test_df[
            "TimeStamp"
        ].min(),
        "~",
        test_df[
            "TimeStamp"
        ].max()
    )

    print(
        "Test rows:",
        len(test_df)
    )

    print(
        "Test defects:",
        int(
            test_df[
                "PassOrFail"
            ].sum()
        )
    )


    # --------------------------------------------------------
    # 빈 test 방지
    # --------------------------------------------------------

    if len(test_df) == 0:

        print(
            "TEST 데이터가 없어서 Skip"
        )

        continue


    X_train = (
        train_df[
            FEATURE_COLS
        ]
    )

    y_train = (
        train_df[
            "PassOrFail"
        ]
        .astype(int)
    )

    X_test = (
        test_df[
            FEATURE_COLS
        ]
    )

    y_test = (
        test_df[
            "PassOrFail"
        ]
        .astype(int)
        .values
    )


    # --------------------------------------------------------
    # 5-model ensemble
    # --------------------------------------------------------

    seed_probabilities = []


    for seed in SEEDS:

        model = LGBMClassifier(
            **BEST_PARAMS,
            class_weight="balanced",
            random_state=seed,
            feature_fraction_seed=seed,
            bagging_seed=seed,
            data_random_seed=seed,
            verbose=-1
        )


        model.fit(
            X_train,
            y_train
        )


        prob = (
            model.predict_proba(
                X_test
            )[:, 1]
        )


        seed_probabilities.append(
            prob
        )


    seed_probabilities = (
        np.column_stack(
            seed_probabilities
        )
    )


    # --------------------------------------------------------
    # Risk / disagreement
    # --------------------------------------------------------

    risk_probability = (
        seed_probabilities
        .mean(axis=1)
    )


    risk_std = (
        seed_probabilities
        .std(axis=1)
    )


    positive_votes = (
        seed_probabilities
        >= RISK_THRESHOLD
    ).sum(axis=1)


    unstable = (
        (positive_votes > 0)
        &
        (
            positive_votes
            < len(SEEDS)
        )
    )


    # --------------------------------------------------------
    # 정책 A: Risk Only
    # --------------------------------------------------------

    risk_only_flag = (
        risk_probability
        >= RISK_THRESHOLD
    ).astype(int)


    # --------------------------------------------------------
    # 정책 B: Risk + Ensemble Disagreement
    # --------------------------------------------------------

    risk_uncertainty_flag = (
        (
            risk_probability
            >= RISK_THRESHOLD
        )
        |
        unstable
    ).astype(int)


    # --------------------------------------------------------
    # PR-AUC
    # --------------------------------------------------------

    if len(np.unique(y_test)) == 2:

        pr_auc = (
            average_precision_score(
                y_test,
                risk_probability
            )
        )

    else:

        pr_auc = np.nan


    print(
        "\nPR-AUC:",
        (
            round(pr_auc, 4)
            if not np.isnan(pr_auc)
            else "N/A"
        )
    )


    # --------------------------------------------------------
    # 정책 성능
    # --------------------------------------------------------

    risk_result = metric_dict(
        y_test,
        risk_only_flag,
        "Risk Only"
    )


    uncertainty_result = metric_dict(
        y_test,
        risk_uncertainty_flag,
        "Risk + Uncertainty"
    )


    risk_result[
        "Fold"
    ] = fold_name

    uncertainty_result[
        "Fold"
    ] = fold_name


    risk_result[
        "PR_AUC"
    ] = pr_auc

    uncertainty_result[
        "PR_AUC"
    ] = pr_auc


    all_policy_results.extend(
        [
            risk_result,
            uncertainty_result
        ]
    )


    comparison = pd.DataFrame(
        [
            risk_result,
            uncertainty_result
        ]
    )


    print(
        "\nPOLICY COMPARISON"
    )

    print(
        comparison[
            [
                "Policy",
                "Precision",
                "Recall",
                "F1",
                "TP",
                "FP",
                "FN",
                "TN",
                "Flag_Count",
                "Flag_Rate_pct"
            ]
        ]
        .round(4)
        .to_string(index=False)
    )


    # --------------------------------------------------------
    # Risk+Uncertainty가 추가한 검사
    # --------------------------------------------------------

    extra_flag = (
        (risk_only_flag == 0)
        &
        (
            risk_uncertainty_flag == 1
        )
    )


    extra_count = int(
        extra_flag.sum()
    )


    extra_defects = int(
        y_test[
            extra_flag
        ].sum()
    )


    fn_reduction = (
        risk_result["FN"]
        -
        uncertainty_result["FN"]
    )


    print(
        "\n추가 검사:",
        extra_count
    )

    print(
        "추가 검사에서 발견한 실제 불량:",
        extra_defects
    )

    print(
        "FN 감소:",
        fn_reduction
    )


    # --------------------------------------------------------
    # 개별 예측 저장
    # --------------------------------------------------------

    pred_df = (
        test_df[
            [
                "TimeStamp",
                "Side",
                "Side_RH",
                "PassOrFail",
                "Episode_ID"
            ]
        ]
        .copy()
    )


    pred_df[
        "Fold"
    ] = fold_name


    pred_df[
        "Risk_Probability"
    ] = risk_probability


    pred_df[
        "Risk_Score"
    ] = (
        risk_probability
        * 100
    )


    pred_df[
        "Risk_STD"
    ] = risk_std


    pred_df[
        "Positive_Votes"
    ] = positive_votes


    pred_df[
        "Prediction_Stability"
    ] = np.where(
        unstable,
        "Unstable",
        "Stable"
    )


    pred_df[
        "Risk_Only_Flag"
    ] = risk_only_flag


    pred_df[
        "Risk_Uncertainty_Flag"
    ] = risk_uncertainty_flag


    all_predictions.append(
        pred_df
    )


    # ========================================================
    # 13. Cycle-level 평가
    #
    # Side 하나라도 실제 불량이면 Cycle 불량
    # Side 하나라도 Flag이면 Cycle 검사
    # ========================================================

    cycle_eval = (
        pred_df
        .groupby(
            "TimeStamp"
        )
        .agg(
            Actual_Cycle=(
                "PassOrFail",
                "max"
            ),

            Risk_Only_Cycle=(
                "Risk_Only_Flag",
                "max"
            ),

            Risk_Uncertainty_Cycle=(
                "Risk_Uncertainty_Flag",
                "max"
            ),

            Max_Risk=(
                "Risk_Probability",
                "max"
            ),

            Episode_ID=(
                "Episode_ID",
                lambda x:
                x.dropna().iloc[0]
                if len(
                    x.dropna()
                ) > 0
                else np.nan
            )
        )
        .reset_index()
    )


    for policy_col, policy_name in [
        (
            "Risk_Only_Cycle",
            "Risk Only"
        ),
        (
            "Risk_Uncertainty_Cycle",
            "Risk + Uncertainty"
        )
    ]:

        cycle_metric = metric_dict(
            cycle_eval[
                "Actual_Cycle"
            ].values,

            cycle_eval[
                policy_col
            ].values,

            policy_name
        )

        cycle_metric[
            "Fold"
        ] = fold_name

        cycle_metric[
            "Evaluation_Level"
        ] = "Cycle"

        cycle_results.append(
            cycle_metric
        )


    print(
        "\nCYCLE-LEVEL POLICY COMPARISON"
    )

    current_cycle_results = (
        pd.DataFrame(
            [
                x
                for x in cycle_results
                if x[
                    "Fold"
                ] == fold_name
            ]
        )
    )


    print(
        current_cycle_results[
            [
                "Policy",
                "Precision",
                "Recall",
                "F1",
                "TP",
                "FP",
                "FN",
                "TN",
                "Flag_Count",
                "Flag_Rate_pct"
            ]
        ]
        .round(4)
        .to_string(index=False)
    )


    # --------------------------------------------------------
    # Target Episode만 별도 출력
    # --------------------------------------------------------

    target_rows = (
        pred_df[
            pred_df[
                "Episode_ID"
            ] == target_episode
        ]
        .copy()
    )


    print(
        f"\n[{target_episode}] FAILURE ROW DETAIL"
    )


    if len(target_rows) == 0:

        print(
            "해당 Episode 없음"
        )

    else:

        print(
            target_rows[
                [
                    "TimeStamp",
                    "Side",
                    "PassOrFail",
                    "Risk_Score",
                    "Risk_STD",
                    "Positive_Votes",
                    "Prediction_Stability",
                    "Risk_Only_Flag",
                    "Risk_Uncertainty_Flag"
                ]
            ]
            .round(
                {
                    "Risk_Score": 4,
                    "Risk_STD": 4
                }
            )
            .to_string(index=False)
        )


# ============================================================
# 14. 전체 Temporal 결과
# ============================================================

policy_results_df = (
    pd.DataFrame(
        all_policy_results
    )
)


cycle_results_df = (
    pd.DataFrame(
        cycle_results
    )
)


temporal_prediction_df = (
    pd.concat(
        all_predictions,
        ignore_index=True
    )
)


print(
    "\n\n"
    + "=" * 120
)

print(
    "FINAL TEMPORAL SIDE-LEVEL RESULTS"
)

print(
    "=" * 120
)

print(
    policy_results_df[
        [
            "Fold",
            "Policy",
            "PR_AUC",
            "Precision",
            "Recall",
            "F1",
            "TP",
            "FP",
            "FN",
            "TN",
            "Flag_Count",
            "Flag_Rate_pct"
        ]
    ]
    .round(4)
    .to_string(index=False)
)


print(
    "\n"
    + "=" * 120
)

print(
    "FINAL TEMPORAL CYCLE-LEVEL RESULTS"
)

print(
    "=" * 120
)

print(
    cycle_results_df[
        [
            "Fold",
            "Policy",
            "Precision",
            "Recall",
            "F1",
            "TP",
            "FP",
            "FN",
            "TN",
            "Flag_Count",
            "Flag_Rate_pct"
        ]
    ]
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 15. 저장
# ============================================================

policy_results_df.to_csv(
    "CN7_P1_9_temporal_policy_results.csv",
    index=False,
    encoding="utf-8-sig"
)


cycle_results_df.to_csv(
    "CN7_P1_9_temporal_cycle_results.csv",
    index=False,
    encoding="utf-8-sig"
)


temporal_prediction_df.to_csv(
    "CN7_P1_9_temporal_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


print(
    "\n저장 완료:"
)

print(
    "CN7_P1_9_temporal_policy_results.csv"
)

print(
    "CN7_P1_9_temporal_cycle_results.csv"
)

print(
    "CN7_P1_9_temporal_predictions.csv"
)

| 구분 | 관찰 결과 | 해석 | 현장 대응 |
|---|---|---|---|
| **확정 모델 사양** | 23개 원본 센서 + `Side_RH` + 직전 5 Cycle 평균 23개 = **47 Features**, LightGBM + class weight | 기존 개발 과정에서 선정한 후보 모델 구조 유지 | 모델 구조 재선정 불필요 |
| **내부 Group-CV** | PR-AUC **0.8157 ± 0.0139** | 기존 failure regime이 포함된 내부 검증에서는 높은 판별력 | **내부 검증 성능**으로 보고 |
| **Side 검증** | Side-only PR-AUC 0.0187 / Side 제거 시 0.6509 / Full 0.8157 | Side 자체가 불량을 결정하는 것이 아니라 공정조건과의 상호작용에 기여 | `RH=불량 원인` 해석 금지 |
| **Rolling Feature** | SHAP 중요도의 약 60%가 Rolling Mean 계열 | 최근 5 Cycle 공정상태가 품질위험 판단에 중요 | 최근 생산이력을 현장 Feature로 유지 |
| **Risk-only 내부 OOF** | Recall **82.35%**, FN 3 | Risk Score만으로 일부 불량 미탐 | Threshold 정책 필요 |
| **Risk+Uncertainty 내부 OOF** | Recall **94.12%**, FN 1 | 내부 OOF에서는 seed disagreement를 추가 검사하면 미탐 감소 | **후보 검사정책**으로 유지 |
| **Uncertainty 한계** | Temporal E4/E5에서는 모든 모델이 낮은 Risk를 내며 disagreement도 거의 발생하지 않음 | 모델들이 함께 틀리는 새로운 regime은 disagreement 방식으로 탐지하지 못함 | Uncertainty만으로 domain shift 대응 불가 |
| **E1~E3** | 센서 패턴 correlation이 높음. E1–E2 0.9120, E2–E3 0.7439 | 10/16 failure들은 상대적으로 유사한 공정 패턴 | 기존 모델이 학습하기 쉬운 failure regime 후보 |
| **E3 vs E4** | Pattern correlation **-0.5566**, 거리 **28.243** | E4는 E3와 공정변수 변화 방향·크기가 크게 다름 | 새로운 regime에 대한 경보 필요  |
| **E4 특징** | Filling Time +17.24σ, Injection Time +10.40σ, Max Injection Speed -11.28σ 등 | 강한 다변량 공정 변화가 나타난 failure c